# Devoir maison 1 — carnets J1 et J2

> **À rendre avant la première séance de la semaine suivante.** Ce devoir porte sur les carnets
> **J1** (du fichier brut à la soumission) et **J2** (ce que la table cache). Il se fait **seul**, à
> la maison, et il se rend.

**Ce qu'on rend :** ce carnet, rempli, déposé dans votre **dépôt GitHub personnel** avec la
déclaration d'usage de l'IA. **Les trois tableaux** de la fin et **l'outil de la question 12** sont
les pièces principales.

**Pourquoi ce devoir :** chaque question demande une mesure que **vous** produisez — jamais une
lecture. Presque toutes s'ouvrent sur **une prédiction écrite avant la mesure** : c'est le geste du
métier, et c'est aussi ce qui rend une erreur visible.

**Quatre mots, pour être sûr :**

- **les deux réponses** — le patient vient (`patient_absent` vaut 0), ou il ne vient pas (1) ;
- **la référence** — le score d'un modèle qui répond toujours la même chose : le minimum à battre ;
- **la barre de bruit** — de combien un score bouge tout seul quand seul le découpage change ;
  en dessous de **deux barres**, un écart n'est pas une différence ;
- **un constat** — une observation, un chiffre, **par rapport à quoi**, et ce que vous en faites.

**Quinze questions, dont douze de cœur.** Chaque titre porte sa mention : **cœur** = le socle du
semestre ; **approfondissement** = à faire dès que le cœur tient. Si le temps manque, faites les
douze de cœur d'abord — les trois approfondissements portent les gestes les plus fins, et ils se
reprennent plus tard. Rien n'est facultatif au sens où tout est lu.

---

## Comment ça marche

1. La partie **« Mise en place »** est **fournie** : exécutez-la de haut en bas. **Deux de ses
   cellules sont faites pour être touchées** : votre **identifiant GitHub** (les questions 2 et 11
   mesurent sur un échantillon tiré de lui), et `ajouter_features` (la question 10).
   **Un identifiant mal recopié change votre échantillon** : copiez-le comme dans l'URL de votre dépôt.
2. Chaque **question** porte sa consigne, en markdown, un **emplacement vide**, et des
   **contrôles de forme** posés dans la cellule : ils disent si votre réponse a la forme attendue
   (un pourcentage, une dimension, une cohérence avec **vos** autres réponses). Ils ne donnent
   **jamais** la valeur attendue, et ils se taisent tant que vous n'avez rien écrit.
3. Les questions sont dans l'ordre des carnets : **1 à 8 sur J1**, **9 à 15 sur J2** — sauf la 15,
   qui transpose la méthode hors de ce jeu de données.
4. Une réponse se **chiffre** ou s'**écrit**. Plusieurs questions demandent d'**annoncer AVANT de
   mesurer** : `noter(...)` refuse d'ailleurs d'enregistrer un essai non annoncé.
5. À la fin : les **trois tableaux**, puis le **dépôt**.

**Ce qui est lu.** **Les trois tableaux** — constats, hypothèses, essais — **et le code**, là où il
produit un chiffre. **Ce dépôt est relevé** : il est lu, et **les sept devoirs comptent ensemble
pour 10 % de la note**, dans la composante « exercices et devoirs » (25 %, avec E1, E2, E3).
**Il n'y a pas d'oral sur les devoirs** : l'oral individuel, en fin de parcours, porte sur
l'ensemble de votre travail — les trois blocs et les deux terrains.

> **La règle, la même qu'en séance :** un score sans référence ne veut rien dire, un écart plus
> petit que deux barres n'est pas une différence, et un constat sans chiffre n'en est pas un.


## Mise en place (fournie)

Ces cellules sont **extraites de `notebook_J1.ipynb` et de `notebook_J2.ipynb`**, qui restent les
sources uniques de la table de travail, de la référence et du modèle. Exécutez-les de haut en bas,
sans rien y modifier — sauf `ajouter_features`, que la question 10 vous demandera d'écrire.

Les `random_state=0` que vous verrez sont la **graine du tirage au sort** (expliquée en J1 §5) :
le même nombre redonne le même découpage, donc la même mesure.


### Les outils


In [1]:
# les outils
# Aucun import en bloc au début : chaque module entre dans le carnet au moment de son premier
# usage. On voit ainsi à quoi il sert, au lieu de le deviner dans une liste.

# `display` : un tableau s'affiche proprement dans un carnet, et reste lisible ailleurs.
# C'est par `display` qu'on montre TOUTES les tables de ce carnet — jamais un print de texte.
try:
    from IPython.display import display
except ImportError:
    def display(x):
        print(x)


### Les données


In [2]:
import os
import pandas as pd
# où sont les fichiers — on CHERCHE `train.csv`, on ne suppose pas un chemin
# (Kaggle a déjà déplacé ses points de montage : un chemin recopié se périme, une recherche non)
def _trouver_dossier(racines):
    """Première racine contenant train.csv — en surface, puis en profondeur."""
    for racine in racines:
        if not os.path.isdir(racine):
            continue
        if os.path.exists(os.path.join(racine, "train.csv")):
            return racine
        for chemin, _, fichiers in os.walk(racine):
            if "train.csv" in fichiers:
                return chemin
    return None

DOSSIER = _trouver_dossier(["/kaggle/input", "sortie"])
assert DOSSIER, "train.csv introuvable : les données de la compétition sont-elles montées ?"


In [3]:
train = pd.read_csv(os.path.join(DOSSIER, "train.csv"), low_memory=False)
test = pd.read_csv(os.path.join(DOSSIER, "test.csv"), low_memory=False)

ID, CIBLE = "ligne_id", "patient_absent"


### La table de travail


In [4]:
A_ECARTER = [ID, "dossier_id", "patient_id", "duree_consultation", "constantes_relevees",
             "motif_absence", "date_prise", "date_rdv", "taille_cm", "date_naissance",
             "canal_prise"]

def matrice(cols):
    """La table de travail : colonnes écartées retirées, catégorielles déclarées."""
    X = cols.drop(columns=[c for c in A_ECARTER + [CIBLE] if c in cols.columns]).copy()
    cat = [c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c])]
    trop_fin = [c for c in cat if X[c].nunique() > 255]
    X = X.drop(columns=trop_fin)
    cat = [c for c in cat if c not in trop_fin]
    for c in cat:
        X[c] = X[c].fillna("").astype("category")
    return X, cat, trop_fin

X, CAT, TROP_FIN = matrice(train)
Xt, _, _ = matrice(test)

for c in CAT:
    Xt[c] = Xt[c].cat.set_categories(X[c].cat.categories)


### La cible, et la référence


In [5]:
repartition = pd.DataFrame({"lignes": [len(train), int(train[CIBLE].sum())],
                            "part (%)": [100.0, round(100 * train[CIBLE].mean(), 2)]},
                           index=["toutes", "absents"])

reference = 100 * max(train[CIBLE].mean(), 1 - train[CIBLE].mean())


In [6]:
y = train[CIBLE]
reference = 100 * max(y.mean(), 1 - y.mean())


### Le premier modèle


In [7]:
# le modèle et le découpage : ils entrent ici
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split
Xtr, Xval, ytr, yval = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)

modele = HistGradientBoostingClassifier(random_state=0).fit(Xtr, ytr)
score_interne = 100 * modele.score(Xval, yval)


### Vos features — LA cellule que vous modifiez


In [8]:
# votre feature : elle reçoit la TABLE DE TRAVAIL — identifiants, dates et colonnes post-issue
# déjà retirés — et la rend enrichie. `predict` l'appelle sur la table BRUTE, après avoir retiré
# les mêmes colonnes : le crochet voit donc la même forme des deux côtés, ce qui est la règle.
def ajouter_features(df):
    """Vos features maison. Appliquée identiquement à l'entraînement et au test."""
    df = df.copy()
    # Exemple — remplacez-le par la vôtre :
    # df["plus_de_60_ans"] = (df["age"] > 60).astype(int)
    return df

# Votre feature entre par `preparer()`, la chaîne unique : on n'applique RIEN à la main.
# Après l'avoir écrite, appelez `remesurer()` (au devoir, dans la cellule prévue) : `X` la
# reçoit une seule fois, et `predict` l'applique aussi, de son côté, sur la table brute.


### La chaîne — une seule, des deux côtés


In [9]:
# ── LA CHAÎNE : une seule fonction, celle que `predict` ET l'entraînement utilisent.
def preparer(brut):
    """Écarte, type, applique vos features : LA chaîne, des deux côtés.
       `predict` l'appelle ; l'entraînement l'appelle aussi, par `matrice()` sur la table de
       travail. Contourner cette fonction, c'est apprendre sur une table et prédire sur une
       autre — exactement le défaut que la séance 2 traque."""
    d = brut.drop(columns=[c for c in A_ECARTER + TROP_FIN + [CIBLE] if c in brut.columns],
                  errors="ignore").copy()
    for c in CAT:
        if c in d.columns:
            d[c] = d[c].fillna("").astype("category").cat.set_categories(X[c].cat.categories)
    return ajouter_features(d)

def remesurer():
    """Réentraîne sur la table de travail COURANTE et met `modele` à jour.
       `ajouter_features` n'est appliquée qu'UNE fois, depuis X : ré-exécutable sans que la
       feature se superpose à elle-même."""
    global modele
    Xc = ajouter_features(X)
    a, b, c, d = train_test_split(Xc, y, test_size=0.2, stratify=y, random_state=0)
    modele = HistGradientBoostingClassifier(random_state=0).fit(a, c)
    return 100 * modele.score(b, d)


### L'interface — le modèle qu'on peut appeler


In [10]:
# numpy, pour la seule assertion qui suit
import numpy as np

def predict(df):
    """Renvoie un tableau de 0/1, un par ligne de df. C'est cette fonction que le cours appellera.
       Elle passe par `preparer(...)`, la même chaîne que l'entraînement — pas une copie."""
    # elle ne refait PAS la chaîne à la main : elle appelle `preparer`, celle de l'entraînement.
    # C'est ce couplage qui garantit qu'une table n'est jamais préparée de deux façons.
    return modele.predict(preparer(df)).astype(int)

def test_predict():
    """Vérifie que l'interface répond ce qu'on attend d'elle."""
    p = predict(test.head(500))

    assert len(p) == 500, f"attendu 500 prédictions, obtenu {len(p)}"
    assert set(np.unique(p)) <= {0, 1}, "les valeurs doivent être 0 ou 1"
    print(f"test OK — 500 prédictions, {int(sum(p))} positifs")

# test_predict()   (au devoir : la cellule « l'interface » s'en charge)


### Le tableau d'essais


In [11]:
# l'horloge entre ici : c'est elle qui rend une prédiction vérifiable
from datetime import datetime

ESSAIS = []
PREDICTIONS = []

def prevoir(essai, attendu, raison):
    """Écrit ce que vous attendez AVANT de mesurer — horodaté.
       `noter(...)` refuse ensuite tout essai sans prédiction : l'ordre n'est pas demandé, il est
       appliqué. Ce que cette fonction garantit, exactement : une prédiction EXISTAIT avant que la
       ligne du tableau soit écrite. Elle ne date pas la mesure — le score lui arrive déjà calculé.
       Pour la garantie complète, mesurez dans `essai(...)`, qui date la mesure lui-même."""
    PREDICTIONS[:] = [p for p in PREDICTIONS if p["essai"] != essai]
    PREDICTIONS.append({"essai": essai, "attendu": attendu, "raison": raison,
                        "écrit à": datetime.now().strftime("%H:%M:%S")})
    display(pd.DataFrame(PREDICTIONS))

def essai(nom, mesure, changement="", attendu=""):
    """Le chemin STRICT : la mesure est prise ICI, donc l'ordre (prédiction puis mesure) est prouvé.
       `mesure` est une fonction sans argument qui rend le score, par exemple
       `lambda: modele.score(Xval, yval)`. Refuse de mesurer sans prédiction."""
    prevu = [p for p in PREDICTIONS if p["essai"] == nom]
    if not prevu:
        print(f"refusé : aucune prédiction pour « {nom} ». Écrivez prevoir(...) d'abord.")
        return None
    score = 100 * mesure()
    noter(nom, changement or "voir la question", attendu or prevu[0]["attendu"], score)
    return score

def noter(hypothese, changement, attendu, score):
    """Ajoute une ligne au tableau. `attendu` : un signe et une raison, jamais une valeur.
       Ré-exécuter une cellule REMPLACE la ligne de même `hypothèse` : elle ne la double pas.

       REFUSE d'enregistrer un essai non annoncé — c'est la règle du cours, appliquée.
       Ce qu'elle garantit, et rien de plus : une prédiction existait AVANT que cette ligne soit
       écrite. Elle ne date pas la mesure — d'où `essai(...)` pour la garantie complète."""
    prevu = [p for p in PREDICTIONS if p["essai"] == hypothese]
    if not prevu:
        print(f"  refusé : aucune prédiction pour « {hypothese} ».")
        print("  Écrivez prevoir(...) d'abord — l'ordre est vérifié, pas seulement demandé.")
        return
    ligne = {"hypothèse": hypothese, "changement": changement,
             "attendu": attendu, "mesuré": round(score, 2),
             "prédiction écrite avant ?": "oui, à " + prevu[0]["écrit à"]}
    ESSAIS[:] = [e for e in ESSAIS if e["hypothèse"] != hypothese] + [ligne]
    display(pd.DataFrame(ESSAIS))

# les deux amorces du carnet sont annoncées elles aussi — sinon elles ne passeraient pas
prevoir("rien faire du tout", "c'est le minimum à battre",
        "un modèle qui répond toujours la même chose vaut la classe majoritaire")
prevoir("un modèle suffit-il ?", "mieux que la référence",
        "les arbres captent une partie du signal, pas tout")


,essai,attendu,raison,écrit à
0,rien faire du tout,c'est le minimum à battre,un modèle qui répond toujours la même chose va...,00:15:14


,essai,attendu,raison,écrit à
0,rien faire du tout,c'est le minimum à battre,un modèle qui répond toujours la même chose va...,00:15:14
1,un modèle suffit-il ?,mieux que la référence,"les arbres captent une partie du signal, pas tout",00:15:14


### La soumission


In [12]:
soumission = pd.DataFrame({ID: test[ID], CIBLE: modele.predict(Xt).astype(int)})
soumission.to_csv("submission.csv", index=False)


### Le modèle linéaire


In [13]:
# le modèle linéaire : il entre ici
from sklearn.linear_model import LogisticRegression
X_num = pd.get_dummies(X, columns=CAT).fillna(0)
Xtr_n, Xval_n = X_num.loc[Xtr.index], X_num.loc[Xval.index]

modele_lineaire = LogisticRegression(solver="newton-cholesky").fit(Xtr_n, ytr)
score_lineaire = 100 * modele_lineaire.score(Xval_n, yval)


### Les outils du devoir : les tableaux, la barre, les contrôles


In [14]:
# les trois tableaux, et les outils qui les remplissent

CONSTATS = []

def constat(observation, chiffre, comparaison, consequence):
    """observation : ce qu'on a vu · chiffre : la mesure exacte
       comparaison : par rapport à quoi ? (un nombre) · conséquence : garder, écarter,
       surveiller, mesurer. Ré-exécuter une cellule REMPLACE la ligne de même `observation`."""
    ligne = {"observation": observation, "chiffre": chiffre,
             "comparaison": comparaison, "conséquence": consequence}
    CONSTATS[:] = [c for c in CONSTATS if c["observation"] != observation] + [ligne]
    display(pd.DataFrame(CONSTATS))

HYPOTHESES = []

def hypothese(feature, ce_qu_elle_raconte, comment_on_la_test, ce_qui_la_contredirait):
    """Une hypothèse est une prédiction qu'on peut tester, avec un critère annoncé AVANT.
       Ré-exécuter une cellule REMPLACE la ligne de même `feature`."""
    ligne = {"feature": feature, "ce qu'elle raconte": ce_qu_elle_raconte,
             "test": comment_on_la_test, "contredite si": ce_qui_la_contredirait}
    HYPOTHESES[:] = [h for h in HYPOTHESES if h["feature"] != feature] + [ligne]
    display(pd.DataFrame(HYPOTHESES))

# les trois constats de la séance 1 — ils comptent : la question 3 en demande CINQ au total,
# donc DEUX des vôtres. Ils sont réécrits ici parce que ce carnet doit tourner seul.
constat("une colonne ne dit rien",
        f"{train['zone_clinique'].nunique()} valeur(s) distincte(s)",
        f"sur les {len(train)} lignes de la table, soit une colonne constante au sens du cours",
        "l'écarter — et ce qu'on y gagne se mesure")
constat("deux colonnes disent la même chose",
        "corrélation 1,000 entre cout_acte et montant_acte",
        f"contre moins de 0,05 pour les autres paires sur {len(train)} lignes : un rapport constant",
        "n'en garder qu'une")
constat("la cible est déséquilibrée",
        f"{100 * train[CIBLE].mean():.2f} % d'absents",
        "contre 50 % pour un problème équilibré : la référence monte d'autant",
        "mesurer : cela conditionne la métrique, donc ce qu'on regarde")
CONSTATS_DE_LA_SEANCE = [c['observation'] for c in CONSTATS]

# les deux amorces de J1 — annoncées, puis notées (c'est la règle de l'ordre, montrée)
noter("rien faire du tout", "classe majoritaire", "c'est le minimum à battre", reference)
noter("un modèle suffit-il ?", "HistGradientBoosting", "mieux que la référence", score_interne)

# la barre de bruit — recalculée ici parce que la question 5 en change la taille du jeu
erreur_type = 100 * ((score_interne / 100) * (1 - score_interne / 100) / len(yval)) ** 0.5
print(f"erreur-type de l'exactitude de validation (jeu de 20 %) : ± {erreur_type:.2f} point(s)")
print("  formule : racine de p(1-p)/n, où n est le nombre de lignes de validation.")
print("  vocabulaire : une barre de bruit EST une erreur-type — « deux barres » = deux erreurs-types.")

# ── VOTRE échantillon : il dépend de VOTRE identifiant GitHub, et de lui seul.
#    Les questions 2 et 11 mesurent dessus : deux voisins n'ont donc pas les mêmes chiffres,
#    et un chiffre recopié ne tombe pas juste. Le reste du devoir (le socle J1/J2) garde le
#    protocole figé du cours — c'est voulu : ce qui doit être comparable entre tous reste
#    comparable, et ce qui doit être VOTRE mesure est la vôtre.
IDENTIFIANT_GITHUB = "…"          # <- remplacez par votre pseudo GitHub, comme dans l'URL
import hashlib
if IDENTIFIANT_GITHUB.strip() and IDENTIFIANT_GITHUB.strip() != '…':
    GRAINE = int(hashlib.sha256(IDENTIFIANT_GITHUB.strip().lower().encode('utf-8')).hexdigest()[:8], 16)
else:
    GRAINE = None
mon_train = train.sample(frac=0.6, random_state=GRAINE) if GRAINE is not None else train
print(f"votre échantillon : {len(mon_train)} lignes sur {len(train)}"
      + ("" if GRAINE is not None else "   (identifiant non renseigné : la table entière)"))

# la table de travail NUE, gardée de côté : c'est sur elle que `remesurer()` rebâtit `X`,
# pour que votre feature s'applique UNE SEULE fois, même si vous relancez la cellule dix fois.
X_sans_features = matrice(train)[0]

def remesurer():
    """Réentraîne sur la table de travail COURANTE et met `modele` à jour — et `X` avec.
       `ajouter_features` n'est appliquée qu'UNE fois, depuis la table nue : ré-exécutable."""
    global modele, X
    X = ajouter_features(X_sans_features)
    a, b, c, d = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)
    modele = HistGradientBoostingClassifier(random_state=0).fit(a, c)
    return 100 * modele.score(b, d)

VERIFICATIONS = []

def verifier(nom, condition, indice=''):
    """Contrôle silencieux : il ne dit JAMAIS la valeur attendue."""
    VERIFICATIONS.append({'contrôle': nom, 'tient': bool(condition)})
    if condition:
        print('  ok        ' + nom)
    else:
        print('  À REVOIR  ' + nom + '   ->   ' + indice)

def controle(nom, quoi, indice='', des_que=None):
    """Contrôle de FORME, posé dans la cellule de la question. Il ne compare JAMAIS à une valeur
       attendue : un type, une plage, une dimension, ou une cohérence avec VOS autres réponses.
       Tant que la réponse n'est pas écrite, il se tait.

       `des_que` : une fonction sans argument qui dit si la réponse EXISTE déjà. Tant qu'elle rend
       False, le contrôle se tait — un contrôle parle d'une réponse écrite, jamais de son absence."""
    if des_que is not None:
        try:
            if not des_que():
                print(f'  (pas encore écrit)  {nom}')
                return
        except NameError:
            print(f'  (pas encore écrit)  {nom}')
            return
    try:
        ok = bool(quoi())
    except NameError:
        print(f'  (pas encore écrit)  {nom}')
        return
    except Exception as e:
        print(f'  REVOIR    {nom}   ->   {type(e).__name__} : {e}')
        return
    VERIFICATIONS.append({'contrôle': nom, 'tient': ok})
    print(('  ok        ' if ok else '  REVOIR    ') + nom + ('' if ok else '   ->   ' + indice))

# le nombre de colonnes de la table AVANT votre feature (la question 10 s'y compare)
n_colonnes_avant = X.shape[1]

# les deux essais du haut du carnet J1 sont des AMORCES : ils ne comptent pas comme les vôtres.
AMORCES = ('rien faire du tout', 'un modèle suffit-il ?')

# la clé de la question 15 vit côté enseignant, pas ici : un vocabulaire de quatre familles,
# c'est 4^6 combinaisons, et une empreinte de nom de colonne se casse en quelques secondes.


,observation,chiffre,comparaison,conséquence
0,une colonne ne dit rien,1 valeur(s) distincte(s),"sur les 149894 lignes de la table, soit une co...",l'écarter — et ce qu'on y gagne se mesure


,observation,chiffre,comparaison,conséquence
0,une colonne ne dit rien,1 valeur(s) distincte(s),"sur les 149894 lignes de la table, soit une co...",l'écarter — et ce qu'on y gagne se mesure
1,deux colonnes disent la même chose,"corrélation 1,000 entre cout_acte et montant_acte","contre moins de 0,05 pour les autres paires su...",n'en garder qu'une


,observation,chiffre,comparaison,conséquence
0,une colonne ne dit rien,1 valeur(s) distincte(s),"sur les 149894 lignes de la table, soit une co...",l'écarter — et ce qu'on y gagne se mesure
1,deux colonnes disent la même chose,"corrélation 1,000 entre cout_acte et montant_acte","contre moins de 0,05 pour les autres paires su...",n'en garder qu'une
2,la cible est déséquilibrée,18.25 % d'absents,contre 50 % pour un problème équilibré : la ré...,"mesurer : cela conditionne la métrique, donc c..."


,hypothèse,changement,attendu,mesuré,prédiction écrite avant ?
0,rien faire du tout,classe majoritaire,c'est le minimum à battre,81.75,"oui, à 00:15:14"


,hypothèse,changement,attendu,mesuré,prédiction écrite avant ?
0,rien faire du tout,classe majoritaire,c'est le minimum à battre,81.75,"oui, à 00:15:14"
1,un modèle suffit-il ?,HistGradientBoosting,mieux que la référence,82.51,"oui, à 00:15:14"


erreur-type de l'exactitude de validation (jeu de 20 %) : ± 0.22 point(s)
  formule : racine de p(1-p)/n, où n est le nombre de lignes de validation.
  vocabulaire : une barre de bruit EST une erreur-type — « deux barres » = deux erreurs-types.
votre échantillon : 149894 lignes sur 149894   (identifiant non renseigné : la table entière)


### L'interface, telle qu'on l'appelle


In [15]:
# `predict` est définie par la mise en place, mais elle est testée ICI : ainsi, `Run All`
# fonctionne même après que vous avez écrit votre feature (la Q9 de l'ancien devoir cassait là).
# Relancez CETTE cellule après avoir touché `ajouter_features` : `X` reçoit la feature une seule
# fois, `modele` est mis à jour, et `predict` l'applique aussi de son côté.
print(f"exactitude de validation, table de travail courante : {remesurer():.2f} %")
test_predict()

import matplotlib.pyplot as plt          # pour les graphiques des questions 2 et 13


exactitude de validation, table de travail courante : 82.51 %
test OK — 500 prédictions, 31 positifs


## Les questions

**Quinze questions : douze de cœur, trois d'approfondissement.** Les huit premières portent sur
**J1**, les sept dernières sur **J2**. Pour chacune : lisez la consigne, écrivez votre code (ou
votre phrase) à l'emplacement prévu, et **exécutez la cellule** — c'est l'exécution qui déclenche
les contrôles.


### Question 1 — La référence, recalculée et prouvée — et le collègue qui annonce 83 %  ·  *carnet J1*  ·  *cœur*


La référence est le chiffre qui décide si tout ce qui suit vaut quelque chose : le score
d'un modèle qui répond **toujours la même chose**. Le carnet l'a calculée sur **tout** `train`.

À faire :
  1. calculez la part d'absents **sur les lignes de contrôle** (`yval`), en **une** ligne, sans
     réutiliser `reference` → `part_absents_controle` ;
  2. **deux chiffres différents, et c'est le piège de la séance** : cette part d'absents n'est **pas**
     la référence. La référence est une **exactitude** — la part de lignes bien classées par un
     modèle qui répond toujours la classe majoritaire (ici : « le patient vient ») : calculez-la
     **sur les lignes de contrôle** → `ma_reference` ;
  3. construisez ce prédicteur : un **tableau de 0/1 de la même longueur que `yval`**, rempli de la
     classe majoritaire — puis montrez qu'il **atteint** `ma_reference` — pas « environ » ;
  4. écrivez l'écart entre `ma_reference` et `reference` (celle de la mise en place, calculée sur
     tout `train`) → `ecart_a_train`, et **pourquoi ce petit écart compte** quand on compare deux
     modèles ;
  5. **un collègue annonce 83 %** d'exactitude. Écrivez votre réponse en **deux chiffres** :
     l'écart à la référence, puis **le nombre de barres de bruit** que vaut cet écart. Un
     score qui ne bat pas la référence de deux barres n'est pas un résultat.

À écrire : `part_absents_controle`, `ma_reference`, `ecart_a_train`, `reponse_au_collegue`,
`ecart_en_barres`.


In [16]:
 import numpy as np

# 1. part d'absents sur les lignes de contrôle (c'est la classe minoritaire)
part_absents_controle = 100 * yval.mean()

# 2. la référence = exactitude d'un modèle qui répond toujours « le patient vient » (0)
ma_reference = 100 * max(yval.mean(), 1 - yval.mean())

# 3. le prédicteur : que des 0, de la même longueur que yval
pred = np.zeros(len(yval), dtype=int)
print("score du prédicteur :", (pred == yval).mean() * 100, "| ma_reference :", ma_reference)

# 4. écart avec la référence calculée sur tout train
ecart_a_train = ma_reference - reference

# 5. le collègue à 83 %
ecart_collegue = 83 - ma_reference
ecart_en_barres = ecart_collegue / erreur_type
reponse_au_collegue = (
    f"83 % bat la référence de {ecart_collegue:.2f} points, soit {ecart_en_barres:.1f} barres "
    f"de bruit (barre = {erreur_type:.2f} point). ..."   # <- termine avec ta conclusion
)
print(ecart_collegue, ecart_en_barres)

controle('la part d absents est un pourcentage, et c est la classe MINORITAIRE',
         lambda: 0 < part_absents_controle < 50,
         'la classe minoritaire : c est elle qu on cherche à détecter')
controle('`ma_reference` est un pourcentage, au-dessus de 50',
         lambda: 50 < ma_reference <= 100,
         'la classe majoritaire fait plus de la moitié : la référence est son score')
controle('les deux chiffres sont complémentaires',
         lambda: abs((part_absents_controle + ma_reference) - 100) < 0.05,
         'part d absents + référence = 100 % : deux faces du même tirage')
controle('le prédicteur ne répond que des 0 et des 1', lambda: set(pred) <= {0, 1})
controle('son score COÏNCIDE avec votre référence',
         lambda: abs((pred == yval).mean() * 100 - ma_reference) < 0.05,
         'les deux mesures doivent tomber ensemble : c est le point de la question')
controle('l écart à `reference` est chiffré', lambda: abs(ecart_a_train) < 5,
         'deux chiffres proches, séparés par un écart de tirage')
controle('la réponse au collègue porte deux chiffres',
         lambda: len(reponse_au_collegue) > 60 and 0 < abs(ecart_en_barres) < 30,
         'écrivez l écart à la référence ET le nombre de barres')


score du prédicteur : 81.75055872444044 | ma_reference : 81.75055872444044
1.2494412755595619 5.694482507311615
  ok        la part d absents est un pourcentage, et c est la classe MINORITAIRE
  ok        `ma_reference` est un pourcentage, au-dessus de 50
  ok        les deux chiffres sont complémentaires
  ok        le prédicteur ne répond que des 0 et des 1
  ok        son score COÏNCIDE avec votre référence
  ok        l écart à `reference` est chiffré
  ok        la réponse au collègue porte deux chiffres


In [17]:
import numpy as np

# 1. part d'absents sur les lignes de contrôle (c'est la classe minoritaire)
part_absents_controle = 100 * yval.mean()

# 2. la référence = exactitude d'un modèle qui répond toujours « le patient vient » (0)
ma_reference = 100 * max(yval.mean(), 1 - yval.mean())

# 3. le prédicteur : que des 0, de la même longueur que yval
pred = np.zeros(len(yval), dtype=int)
print("score du prédicteur :", (pred == yval).mean() * 100, "| ma_reference :", ma_reference)

# 4. écart avec la référence calculée sur tout train
ecart_a_train = ma_reference - reference

# 5. le collègue à 83 %
ecart_collegue = 83 - ma_reference
ecart_en_barres = ecart_collegue / erreur_type
reponse_au_collegue = (
    f"83 % bat la référence de {ecart_collegue:.2f} points, soit {ecart_en_barres:.1f} barres "
    f"de bruit (barre = {erreur_type:.2f} point). ..."   # <- termine avec ta conclusion
)
print(ecart_collegue, ecart_en_barres)

score du prédicteur : 81.75055872444044 | ma_reference : 81.75055872444044
1.2494412755595619 5.694482507311615


### Question 2 — Le balayage, le piège des valeurs rares, et le critère des 500 lignes  ·  *carnet J1*  ·  *cœur*


Trouvez, en regroupant, la colonne qui **sépare le mieux** les deux réponses. Un premier
balayage, **sans aucun filtre**, va désigner une colonne très probablement inutile.

**Tout se fait sur `mon_train` — VOTRE échantillon**, celui que la mise en place a tiré à partir
de votre identifiant GitHub : vos chiffres ne sont pas ceux de votre voisin.

À faire :
  1. pour **toutes** les colonnes de `mon_train`, groupez **par cette colonne** et prenez la
     **moyenne de la cible** dans chaque groupe : c'est le **taux d'absence** du groupe, la colonne
     `taux_absence` de la séance 1. L'écart cherché est celui entre le taux le plus haut et le taux
     le plus bas — sans filtre, et triez ;
  2. notez **laquelle gagne** et son écart : c'est `colonne_piegee` et `ecart_non_filtre`. Comptez
     combien de ses groupes atteignent 500 lignes : c'est `groupes_de_500` ;
  3. **puis** appliquez le critère du cours : **ne gardez que les groupes d'au moins 500 lignes, et
     comparez ceux-là** — refaites le classement (`classement`), et notez le plus petit groupe gardé
     (`plus_petit_groupe_du_classement`) ;
  4. `meilleure_colonne` et son écart `ecart_filtre` ; puis `ce_que_le_critere_ne_protege_pas` : le
     critère a-t-il suffi à écarter le piège ? Regardez ce qui monte en tête du classement, et
     demandez-vous **ce qu'il faudrait vérifier en plus** du critère ;
  5. `lecture` : pourquoi la colonne du premier balayage gagnait-elle ? par rapport à quoi ?

À écrire : `balayage_brut`, `colonne_piegee`, `ecart_non_filtre`, `groupes_de_500`, `classement`,
`plus_petit_groupe_du_classement`, `meilleure_colonne`, `ecart_filtre`,
`ce_que_le_critere_ne_protege_pas`, `lecture`.


In [18]:
# TODO : le balayage SANS filtre, sur toutes les colonnes de `mon_train` (votre échantillon).
# balayage_brut = … (à vous : une ligne par colonne, l'écart max-min des taux)
# colonne_piegee = … (à vous)
# ecart_non_filtre = … (à vous : en points)
# groupes_de_500 = … (à vous : combien de ses groupes atteignent 500 lignes)
# TODO : le balayage FILTRÉ — ne garder que les groupes de 500 lignes et plus, PUIS comparer.
# classement = … (à vous)
# plus_petit_groupe_du_classement = … (à vous : le plus petit groupe que vous avez GARDÉ)
# meilleure_colonne = … (à vous)
# ecart_filtre = … (à vous : en points)
# ce_que_le_critere_ne_protege_pas = … (à vous : que reste-t-il en tête, et pourquoi)
# lecture = … (à vous : pourquoi la première gagnait-elle ? et par rapport à quoi ?)

controle('le balayage couvre toutes les colonnes', lambda: len(balayage_brut) >= 30,
         'une ligne par colonne de `train`, valeurs d identifiant comprises')
controle('l écart non filtré est de l ordre de 100 points',
         lambda: 50 < ecart_non_filtre <= 100, 'le piège donne un écart énorme')
controle('la colonne piégée n a AUCUN groupe de 500 lignes',
         lambda: groupes_de_500 == 0,
         'chaque ligne a son propre groupe : ce n est pas une trouvaille')
controle('le classement filtré est plus petit que le non filtré',
         lambda: len(classement) < len(balayage_brut),
         'le critère des 500 lignes retire des colonnes')
controle('la meilleure colonne est bien celle de VOTRE classement',
         lambda: meilleure_colonne == classement.index[0],
         'la première de votre propre classement — l écart est net, pas d hésitation')
controle('le classement filtré ne garde QUE des groupes de 500 lignes et plus',
         lambda: plus_petit_groupe_du_classement >= 500,
         'le critère se pose AVANT la comparaison : ne gardez que les gros groupes')
controle('les deux écarts sont chiffrés en POINTS, et le filtré est plus petit',
         lambda: 0 < ecart_filtre <= 100 and ecart_filtre < ecart_non_filtre,
         'en points (0 à 100), pas en proportion')
controle('la lecture compare, et le critère est interrogé',
         lambda: len(lecture) > 60 and len(ce_que_le_critere_ne_protege_pas) > 60,
         'par rapport à quoi ? et le critère des 500 lignes suffit-il ?')


  (pas encore écrit)  le balayage couvre toutes les colonnes
  (pas encore écrit)  l écart non filtré est de l ordre de 100 points
  (pas encore écrit)  la colonne piégée n a AUCUN groupe de 500 lignes
  (pas encore écrit)  le classement filtré est plus petit que le non filtré
  (pas encore écrit)  la meilleure colonne est bien celle de VOTRE classement
  (pas encore écrit)  le classement filtré ne garde QUE des groupes de 500 lignes et plus
  (pas encore écrit)  les deux écarts sont chiffrés en POINTS, et le filtré est plus petit
  (pas encore écrit)  la lecture compare, et le critère est interrogé


In [19]:
def balayage(df, min_lignes=None):
    """Écart max-min du taux d'absence entre les groupes de chaque colonne (en points)."""
    res = {}
    for c in df.columns:
        if c == CIBLE:
            continue
        g = df.groupby(c)[CIBLE].agg(["mean", "size"])
        if min_lignes:
            g = g[g["size"] >= min_lignes]
        if len(g) >= 2:
            res[c] = 100 * (g["mean"].max() - g["mean"].min())
    return pd.Series(res).sort_values(ascending=False)

balayage_brut = balayage(mon_train)                       # SANS filtre
colonne_piegee = balayage_brut.index[0]
ecart_non_filtre = balayage_brut.iloc[0]
tailles = mon_train.groupby(colonne_piegee).size()
groupes_de_500 = int((tailles >= 500).sum())

classement = balayage(mon_train, 500)                     # AVEC filtre : groupes >= 500 lignes
_plus_petits = []
for c in classement.index:
    s = mon_train.groupby(c).size()
    _plus_petits.append(s[s >= 500].min())
plus_petit_groupe_du_classement = int(min(_plus_petits))
meilleure_colonne = classement.index[0]
ecart_filtre = classement.iloc[0]

print(balayage_brut.head(5)); print(classement.head(5))

ce_que_le_critere_ne_protege_pas = (
    f"Le critère des 500 lignes écarte {colonne_piegee}, mais il ne dit pas si la colonne "
    f"en tête ({meilleure_colonne}) est utilisable : elle peut être connue seulement après "
    "l'issue (fuite) ou disparaître en production. Il faut vérifier ce que la colonne "
    "VEUT DIRE et le moment où elle existe, pas seulement la taille des groupes."
)
lecture = (
    f"{colonne_piegee} gagnait avec {ecart_non_filtre:.1f} points parce que chacun de ses "
    "groupes est minuscule (souvent une seule ligne) : un taux vaut alors 0 % ou 100 % par "
    f"hasard. Par rapport à {ecart_filtre:.1f} points une fois les groupes de 500 lignes exigés, "
    "l'écart n'était pas une trouvaille mais un artefact de la taille des groupes."
)

dossier_id               100.0
ligne_id                 100.0
patient_id               100.0
taille_cm                100.0
indice_risque_externe    100.0
dtype: float64
duree_consultation    99.400480
canal_prise           42.441885
taux_absence_passe    29.214319
delai_jours           25.133653
clinique              23.258234
dtype: float64


### Question 3 — Cinq constats chiffrés — trois de la séance, deux de vous  ·  *carnet J1*  ·  *cœur*


Cinq constats, **dont au moins deux mesurés par vous**. Les trois premiers sont ceux de la
séance 1 : ils sont déjà écrits par la mise en place, ne les réécrivez pas.

À faire : ajoutez **deux** constats avec `constat(...)` — trois de la séance plus deux des vôtres,
cinq au total — et écrivez-les pour qu'un programme puisse les vérifier, c'est ce qu'il fait :
  · `chiffre` et `comparaison` doivent contenir **un nombre** (un taux, un effectif, un écart) ;
  · `conséquence` doit venir du vocabulaire du cours : **garder**, **écarter**, **surveiller**,
    **mesurer** ;
  · au moins un de vos constats doit être **négatif** — un défaut cherché et non trouvé, ou un
    écart nul. C'est un résultat, pas un échec ;
  · au moins un doit **changer une décision** (garder ou écarter une colonne).

À écrire : les constats, via `constat(...)`.


In [20]:
# TODO : vos DEUX constats (trois de la séance + deux des vôtres = cinq).
# constat("ce que j'ai remarqué", "le chiffre", "par rapport à quoi, avec un nombre",
#        "garder / écarter / surveiller / mesurer")

import re

controle('cinq constats au total', lambda: len(CONSTATS) == 5,
         des_que=lambda: len(CONSTATS) > len(CONSTATS_DE_LA_SEANCE),
         )
controle('au moins deux constats sont les VÔTRES',
         lambda: sum(1 for c in CONSTATS if c['observation'] not in CONSTATS_DE_LA_SEANCE) >= 2,
         'les trois constats de la séance 1 ne comptent pas comme les vôtres',
         des_que=lambda: len(CONSTATS) > len(CONSTATS_DE_LA_SEANCE))
controle('chaque `chiffre` porte un nombre',
         lambda: all(re.search(r'\d', str(c['chiffre'])) for c in CONSTATS),
         'une mesure exacte : un taux, un effectif, un écart')
controle('chaque `comparaison` porte un nombre',
         lambda: all(re.search(r'\d', str(c['comparaison'])) for c in CONSTATS),
         'par rapport à QUOI : un nombre, pas une impression')
controle('chaque `conséquence` vient du vocabulaire du cours',
         lambda: all(any(m in str(c['conséquence']).lower()
                         for m in ('garder', 'ecarter', 'écarter', 'surveiller', 'mesurer'))
                     for c in CONSTATS),
         'garder, écarter, surveiller ou mesurer — pas autre chose')
controle('au moins un constat négatif',
         lambda: any(m in ' '.join(str(c['observation']) + ' ' + str(c['chiffre'])
                                   for c in CONSTATS).lower()
                     for m in ('aucun', 'aucune', 'rien', 'zéro', 'zero', '0 ', 'nul')),
         'un défaut cherché et non trouvé est un résultat : écrivez-le')
controle('au moins un constat change une décision',
         lambda: any(('garder' in str(c['conséquence']).lower()
                      or 'ecarter' in str(c['conséquence']).lower()
                      or 'écarter' in str(c['conséquence']).lower()) for c in CONSTATS),
         'au moins un constat doit dire ce qu on garde ou ce qu on écarte')


  (pas encore écrit)  cinq constats au total
  (pas encore écrit)  au moins deux constats sont les VÔTRES
  ok        chaque `chiffre` porte un nombre
  ok        chaque `comparaison` porte un nombre
  ok        chaque `conséquence` vient du vocabulaire du cours
  ok        au moins un constat négatif
  ok        au moins un constat change une décision


In [21]:
# Deux constats À VOUS (les trois de la séance sont déjà écrits). Il faut en tout 5.
n_dup = int(mon_train.drop(columns=[ID]).duplicated().sum())
constat(f"{'aucun' if n_dup == 0 else 'des'} doublon exact (hors identifiant)",
        f"{n_dup} ligne(s) dupliquée(s)",
        f"sur {len(mon_train)} lignes, soit {100 * n_dup / len(mon_train):.2f} %",
        "mesurer le coût d'un nettoyage avant de l'imposer")

n_neg = int((train["age"] < 0).sum())
constat("un âge impossible (valeur -1) dans age",
        f"{n_neg} ligne(s) avec age < 0",
        f"sur {len(train)} lignes, soit {100 * n_neg / len(train):.2f} % de la table",
        "garder la colonne mais traiter la sentinelle comme une valeur manquante")
# (ajoute les controle(...) du sujet)

,observation,chiffre,comparaison,conséquence
0,une colonne ne dit rien,1 valeur(s) distincte(s),"sur les 149894 lignes de la table, soit une co...",l'écarter — et ce qu'on y gagne se mesure
1,deux colonnes disent la même chose,"corrélation 1,000 entre cout_acte et montant_acte","contre moins de 0,05 pour les autres paires su...",n'en garder qu'une
2,la cible est déséquilibrée,18.25 % d'absents,contre 50 % pour un problème équilibré : la ré...,"mesurer : cela conditionne la métrique, donc c..."
3,aucun doublon exact (hors identifiant),0 ligne(s) dupliquée(s),"sur 149894 lignes, soit 0.00 %",mesurer le coût d'un nettoyage avant de l'imposer


,observation,chiffre,comparaison,conséquence
0,une colonne ne dit rien,1 valeur(s) distincte(s),"sur les 149894 lignes de la table, soit une co...",l'écarter — et ce qu'on y gagne se mesure
1,deux colonnes disent la même chose,"corrélation 1,000 entre cout_acte et montant_acte","contre moins de 0,05 pour les autres paires su...",n'en garder qu'une
2,la cible est déséquilibrée,18.25 % d'absents,contre 50 % pour un problème équilibré : la ré...,"mesurer : cela conditionne la métrique, donc c..."
3,aucun doublon exact (hors identifiant),0 ligne(s) dupliquée(s),"sur 149894 lignes, soit 0.00 %",mesurer le coût d'un nettoyage avant de l'imposer
4,un âge impossible (valeur -1) dans age,46 ligne(s) avec age < 0,"sur 149894 lignes, soit 0.03 % de la table",garder la colonne mais traiter la sentinelle c...


### Question 4 — Trois défauts établis — une prédiction chiffrée, puis la même sur le modèle linéaire  ·  *carnet J1*  ·  *cœur*


Trois défauts sont **établis** : un nombre rangé en texte, des lignes dupliquées, un âge
impossible. Un diagnostic **certain** ne dit rien de ce qu'il **coûte**.

À faire :
  1. **AVANT de mesurer** : pour chacun des trois, écrivez ce que vous attendez — le score
     bouge-t-il, et **de combien de barres de bruit** ? (`pred_age`, `pred_doublons`,
     `pred_taille`, en barres) ;
  2. mesurez la correction de l'âge sur l'ensemble d'arbres : `s_avec_age`, `s_sans_age`, et
     l'écart en barres (`age_en_barres`) ;
  3. **faites la MÊME correction sur le modèle linéaire** (`modele_lineaire` est dans la mise
     en place) : `lin_avec_age`, `lin_sans_age`, `lin_age_en_barres`. Dites ce que devient un
     âge à −1 pour un arbre, et ce qu'il devient pour une somme pondérée ;
  4. `verdict_age` : au vu de **vos deux mesures**, ce défaut compte-t-il — et pour qui ?
     Écrivez votre réponse, et ce qui la distinguerait du verdict qu'appellerait un défaut d'un
     autre effet. Puis `verdict_doublons` : que vaut un contrôle qui ne trouve **rien** ?

À écrire : les trois prédictions en barres, les deux mesures, les deux verdicts.


In [22]:
# TODO : les trois prédictions, AVANT toute mesure (en barres de bruit).
# pred_age = … (à vous)
# pred_doublons = … (à vous)
# pred_taille = … (à vous)
# TODO : l'âge sur les arbres, puis sur le modèle linéaire (pourcentages).
# s_avec_age = … (à vous)
# s_sans_age = … (à vous)
# age_en_barres = … (à vous : l'écart divisé par la barre)
# lin_avec_age = … (à vous)
# lin_sans_age = … (à vous)
# lin_age_en_barres = … (à vous)
# TODO : les deux verdicts, en deux phrases.
# verdict_age = … (à vous)
# verdict_doublons = … (à vous)

controle('les trois prédictions sont écrites avant toute mesure',
         lambda: abs(pred_age) < 20 and abs(pred_doublons) < 20 and abs(pred_taille) < 20,
         'trois nombres, en barres de bruit')
controle('deux scores d arbres, en pourcentage',
         lambda: 50 < s_avec_age <= 100 and 50 < s_sans_age <= 100)
controle('deux scores linéaires, en pourcentage',
         lambda: 50 < lin_avec_age <= 100 and 50 < lin_sans_age <= 100)
controle('l écart des arbres et celui du linéaire sont chiffrés',
         lambda: abs(age_en_barres) < 10 and abs(lin_age_en_barres) < 10,
         'deux écarts, chacun divisé par la barre')
controle('les deux verdicts sont argumentés',
         lambda: len(verdict_age) > 60 and len(verdict_doublons) > 40,
         'dites pour qui le défaut compte, mesures à l appui')


  (pas encore écrit)  les trois prédictions sont écrites avant toute mesure
  (pas encore écrit)  deux scores d arbres, en pourcentage
  (pas encore écrit)  deux scores linéaires, en pourcentage
  (pas encore écrit)  l écart des arbres et celui du linéaire sont chiffrés
  (pas encore écrit)  les deux verdicts sont argumentés


In [23]:
# 1) PRÉDICTIONS, écrites AVANT toute mesure (en barres de bruit) — mets TES valeurs :
pred_age = 0.5        # un arbre isole -1 d'un seul seuil : effet faible attendu
pred_doublons = 0.0   # contrôle trouvé vide : rien à corriger
pred_taille = 0.0     # taille_cm est déjà dans A_ECARTER : aucun effet

from sklearn.ensemble import HistGradientBoostingClassifier as HGB
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

# "avec_age" = âge tel quel (avec le -1) ; "sans_age" = âge corrigé (sans le défaut)
Xa = X_sans_features.copy()
Xb = X_sans_features.copy(); Xb.loc[Xb["age"] < 0, "age"] = np.nan

def score_arbre(Xc):
    a, b, c, d = train_test_split(Xc, y, test_size=0.2, stratify=y, random_state=0)
    return 100 * HGB(random_state=0).fit(a, c).score(b, d)

s_avec_age = score_arbre(Xa)
s_sans_age = score_arbre(Xb)
age_en_barres = (s_sans_age - s_avec_age) / erreur_type

# même correction sur le modèle linéaire : on remplace le -1 par la médiane des âges valides
def score_lin(Xc):
    Xn = pd.get_dummies(Xc, columns=CAT).fillna(0)
    a, b, c, d = train_test_split(Xn, y, test_size=0.2, stratify=y, random_state=0)
    return 100 * LogisticRegression(solver="newton-cholesky").fit(a, c).score(b, d)

Xc_lin = X_sans_features.copy()
med = Xc_lin.loc[Xc_lin["age"] >= 0, "age"].median()
Xc_lin2 = Xc_lin.copy(); Xc_lin2.loc[Xc_lin2["age"] < 0, "age"] = med
lin_avec_age = score_lin(Xc_lin)
lin_sans_age = score_lin(Xc_lin2)
lin_age_en_barres = (lin_sans_age - lin_avec_age) / erreur_type

print(s_avec_age, s_sans_age, age_en_barres)
print(lin_avec_age, lin_sans_age, lin_age_en_barres)

verdict_age = (
    f"Arbres : {age_en_barres:.2f} barre(s) ; linéaire : {lin_age_en_barres:.2f} barre(s). "
    "Pour un arbre, -1 est juste une valeur de plus qu'un seuil isole ; pour une somme "
    "pondérée, -1 est lu comme un âge réel et tire le coefficient. Le défaut compte donc "
    "surtout pour le modèle linéaire (à confirmer par TES chiffres : < 2 barres = dans le bruit)."
)
verdict_doublons = (
    "Un contrôle qui ne trouve rien est un résultat : il établit l'absence du défaut "
    "(0 ligne dupliquée), donc rien à corriger, et on peut le noter comme constat négatif."
)
controle('les trois prédictions sont écrites avant toute mesure',
         lambda: abs(pred_age) < 20 and abs(pred_doublons) < 20 and abs(pred_taille) < 20,
         'trois nombres, en barres de bruit')
controle('deux scores d arbres, en pourcentage',
         lambda: 50 < s_avec_age <= 100 and 50 < s_sans_age <= 100)
controle('deux scores linéaires, en pourcentage',
         lambda: 50 < lin_avec_age <= 100 and 50 < lin_sans_age <= 100)
controle('l écart des arbres et celui du linéaire sont chiffrés',
         lambda: abs(age_en_barres) < 10 and abs(lin_age_en_barres) < 10,
         'deux écarts, chacun divisé par la barre')
controle('les deux verdicts sont argumentés',
         lambda: len(verdict_age) > 60 and len(verdict_doublons) > 40,
         'dites pour qui le défaut compte, mesures à l appui')


82.50775542880015 82.49441275559558 -0.060810876549766246
82.2075452816972 82.2075452816972 0.0
  ok        les trois prédictions sont écrites avant toute mesure
  ok        deux scores d arbres, en pourcentage
  ok        deux scores linéaires, en pourcentage
  ok        l écart des arbres et celui du linéaire sont chiffrés
  ok        les deux verdicts sont argumentés


### Question 5 — La barre décroît comme 1/racine(n) — vos propres tirages  ·  *carnet J1*  ·  *cœur*


L'entraînement ne change pas. On ne change que la **taille du jeu de contrôle**, et on
regarde de combien le score bouge **tout seul**.

À faire :
  1. gardez le modèle de la mise en place (`modele`), et le jeu de contrôle `Xval`/`yval` ;
  2. tirez **20 fois** 1 000 lignes de `Xval`, mesurez chaque fois (`ecarts_1000`) ; refaites-le
     pour 5 000 lignes (`ecarts_5000`) ;
  3. calculez l'**écart-type** de vos 20 scores pour chaque taille (`ecart_type_1000`,
     `ecart_type_5000`) ;
  4. calculez la barre **annoncée par la formule du cours**, racine de p(1-p)/n, pour chaque
     taille (`formule_1000`, `formule_5000`), et le **rapport** observé/formule
     (`rapport_1000`, `rapport_5000`) ;
  5. `lecture` : dans quel sens la barre bouge quand `n` est multiplié par 5 ? Le rapport
     vaut-il à peu près 1 ? Et **vos 20 tirages suffisent-ils** pour estimer un écart-type ?

À écrire : les deux listes de 20 scores, les deux écarts-types, les deux formules, les deux
rapports, et la lecture.


In [24]:
# TODO : 20 tirages de 1 000 lignes, puis 20 tirages de 5 000 lignes, dans Xval.
# ecarts_1000 = … (à vous)
# ecarts_5000 = … (à vous)
# ecart_type_1000 = … (à vous)
# ecart_type_5000 = … (à vous)
# formule_1000 = … (à vous : racine de p(1-p)/n)
# formule_5000 = … (à vous)
# rapport_1000 = … (à vous : observé divisé par la formule)
# rapport_5000 = … (à vous)
# lecture = … (à vous)

controle('vingt tirages à chaque taille',
         lambda: len(ecarts_1000) == 20 and len(ecarts_5000) == 20)
controle('les scores tirés sont des pourcentages',
         lambda: 50 < min(ecarts_1000) and max(ecarts_1000) <= 100
                 and 50 < min(ecarts_5000) and max(ecarts_5000) <= 100)
controle('les deux barres ont été mesurées SÉPARÉMENT',
         lambda: abs(ecart_type_1000 - ecart_type_5000) > 1e-9,
         'deux tailles, deux mesures : ne recopiez pas l une dans l autre')
controle('les deux barres sont chiffrées',
         lambda: 0 < ecart_type_1000 < 10 and 0 < ecart_type_5000 < 10 and formule_1000 > 0)
controle('les deux rapports observé / formule sont chiffrés',
         lambda: 0.2 < rapport_1000 < 5 and 0.2 < rapport_5000 < 5,
         'deux nombres : ce que vous avez mesuré, divisé par ce que la formule annonce')
controle('la lecture parle du nombre de tirages',
         lambda: len(lecture) > 60,
         'la barre, le rapport, et la qualité de votre estimation')


  (pas encore écrit)  vingt tirages à chaque taille
  (pas encore écrit)  les scores tirés sont des pourcentages
  (pas encore écrit)  les deux barres ont été mesurées SÉPARÉMENT
  (pas encore écrit)  les deux barres sont chiffrées
  (pas encore écrit)  les deux rapports observé / formule sont chiffrés
  ok        la lecture parle du nombre de tirages


In [25]:
import numpy as np
from sklearn.ensemble import HistGradientBoostingClassifier as HGB

# modèle de la mise en place, refait à l'identique
modele_base = HGB(random_state=0).fit(Xtr, ytr)
ok = (modele_base.predict(Xval) == yval.values)   # True si bien classé
p = ok.mean()
rng = np.random.default_rng(0)

# 20 tirages de 1 000 lignes, puis 20 tirages de 5 000 lignes (scores en %)   corrige moi ca alors
ecarts_1000 = [100 * ok[rng.choice(len(ok), 1000, replace=False)].mean() for _ in range(20)]
ecarts_5000 = [100 * ok[rng.choice(len(ok), 5000, replace=False)].mean() for _ in range(20)]

# écart-type observé
ecart_type_1000 = float(np.std(ecarts_1000, ddof=1))
ecart_type_5000 = float(np.std(ecarts_5000, ddof=1))

# barre annoncée par la formule : racine de p(1-p)/n
formule_1000 = 100 * (p * (1 - p) / 1000) ** 0.5
formule_5000 = 100 * (p * (1 - p) / 5000) ** 0.5

# rapport observé / formule
rapport_1000 = ecart_type_1000 / formule_1000
rapport_5000 = ecart_type_5000 / formule_5000

print("1000 lignes :", round(ecart_type_1000, 3), round(formule_1000, 3), round(rapport_1000, 2))
print("5000 lignes :", round(ecart_type_5000, 3), round(formule_5000, 3), round(rapport_5000, 2))

# la lecture : la partie sur le rapport suit vos mesures proche = (0.7 < rapport_1000 < 1.3) and (0.7 < rapport_5000 < 1.3)
phrase_rapport = (
    "valent à peu près 1 : la formule décrit bien le bruit observé."
    if proche
    else "s'écartent de 1 : la formule n'est qu'approchée ici, ce qui peut venir du faible nombre de tirages."
)

lecture = (
    f"Quand n est multiplié par 5, la barre est divisée par environ racine de 5 (≈2,2) : "
    f"{ecart_type_1000:.2f} -> {ecart_type_5000:.2f}. Les rapports observé/formule "
    f"({rapport_1000:.2f} et {rapport_5000:.2f}) " + phrase_rapport +
    " Mais 20 tirages ne donnent qu'une estimation grossière d'un écart-type (de l'ordre de 15 % "
    "d'incertitude) : les rapports peuvent s'écarter de 1 sans contredire la formule."
)
print(lecture)

# ---- Les contrôles du sujet (ne pas modifier) ----
controle('vingt tirages à chaque taille',
         lambda: len(ecarts_1000) == 20 and len(ecarts_5000) == 20)
controle('les scores tirés sont des pourcentages',
         lambda: 50 < min(ecarts_1000) and max(ecarts_1000) <= 100
                 and 50 < min(ecarts_5000) and max(ecarts_5000) <= 100)
controle('les deux barres ont été mesurées SÉPARÉMENT',
         lambda: abs(ecart_type_1000 - ecart_type_5000) > 1e-9,
         'deux tailles, deux mesures : ne recopiez pas l une dans l autre')
controle('les deux barres sont chiffrées',
         lambda: 0 < ecart_type_1000 < 10 and 0 < ecart_type_5000 < 10 and formule_1000 > 0)
controle('les deux rapports observé / formule sont chiffrés',
         lambda: 0.2 < rapport_1000 < 5 and 0.2 < rapport_5000 < 5,
         'deux nombres : ce que vous avez mesuré, divisé par ce que la formule annonce')
controle('la lecture parle du nombre de tirages',
         lambda: len(lecture) > 60,
         'la barre, le rapport, et la qualité de votre estimation')

1000 lignes : 1.251 1.201 1.04
5000 lignes : 0.401 0.537 0.75


NameError: name 'proche' is not defined

### Question 6 — La formule qui déborde : ce qu'on ne peut plus tirer  ·  *carnet J1*  ·  *approfondissement*


Reprenez le même tirage, mais avec **25 000 lignes** — presque tout le jeu de contrôle.

À faire :
  1. tirez 20 fois 25 000 lignes, calculez l'écart-type (`ecart_type_25000`) ;
  2. calculez la barre de la formule (`formule_25000`) et le rapport
     (`rapport_25000`) ;
  3. `pourquoi_la_formule_deborde` : en une phrase, ce que la formule suppose et que ce tirage-là
     ne fait plus. Comparez les trois rapports que vous venez de calculer (1 000, 5 000, 25 000) :
     que vaut chacun, et qu'est-ce qui a changé entre le deuxième et le troisième ?
  4. **bonus si vous le voulez** : la **correction de population finie** — l'écart-type d'un tirage
     **sans remise** dans un jeu de `N` lignes se multiplie par racine de (N-n)/(N-1). Calculez-la
     pour n = 25 000, et regardez le rapport qu'elle donne.

À écrire : les 20 scores, l'écart-type, la formule, le rapport, et l'explication.


In [ ]:
# TODO : 20 tirages de 25 000 lignes dans Xval.
# ecarts_25000 = … (à vous)
# ecart_type_25000 = … (à vous)
# formule_25000 = … (à vous)
# rapport_25000 = … (à vous : il doit être nettement plus petit que 1)
# pourquoi_la_formule_deborde = … (à vous)

controle('vingt tirages de 25 000 lignes', lambda: len(ecarts_25000) == 20,
         'la taille demandée est-elle compatible avec celle du jeu de contrôle ?')
controle('la barre observée et la formule ont été chiffrées séparément',
         lambda: 0 < ecart_type_25000 < 10 and 0 < formule_25000 < 10,
         'deux nombres : la mesure observée, puis la formule — pas l un recopié dans l autre')
controle('le rapport est un nombre positif', lambda: 0 < rapport_25000 < 10)
controle('l explication tient en une phrase complète',
         lambda: len(pourquoi_la_formule_deborde) > 60,
         'la formule suppose des tirages indépendants — et là ?')


In [ ]:
import numpy as np
from sklearn.ensemble import HistGradientBoostingClassifier as HGB

# modèle de la mise en place, refait à l'identique
modele_base = HGB(random_state=0).fit(Xtr, ytr)
ok = (modele_base.predict(Xval) == yval.values)   # True si bien classé
p = ok.mean()
N = len(ok)
n6 = min(25000, N - 1)
rng6 = np.random.default_rng(6)

# 20 tirages de 25 000 lignes (scores en %)
ecarts_25000 = [100 * ok[rng6.choice(N, n6, replace=False)].mean() for _ in range(20)]
ecart_type_25000 = float(np.std(ecarts_25000, ddof=1))
formule_25000 = 100 * (p * (1 - p) / n6) ** 0.5
rapport_25000 = ecart_type_25000 / formule_25000

# bonus : correction de population finie (tirage sans remise)
correction = ((N - n6) / (N - 1)) ** 0.5
formule_corrigee = formule_25000 * correction
rapport_corrige = ecart_type_25000 / formule_corrigee

print("25000 lignes :", round(ecart_type_25000, 3), round(formule_25000, 3), round(rapport_25000, 2))
print("bonus : correction", round(correction, 3), "| formule corrigée", round(formule_corrigee, 3), "| rapport corrigé", round(rapport_corrige, 2))
print("rappel : rapport 1000 =", round(rapport_1000, 2), "| rapport 5000 =", round(rapport_5000, 2))

# l'explication : la phrase sur le rapport suit vos mesures
phrase_rapport = (
    "Entre le deuxième et le troisième, le rapport chute : la formule surestime la barre. "
    if rapport_25000 < 0.8
    else "Le rapport reste proche de 1 : ici la formule tient encore. "
)

pourquoi_la_formule_deborde = (
    "La formule suppose des tirages indépendants (comme dans un très grand jeu, avec remise). "
    f"Ici on tire {n6} lignes sur {N} sans remise : les tirages se recouvrent presque entièrement "
    "et se ressemblent, donc le score bouge moins que prévu. "
    f"Rapports observé/formule : {rapport_1000:.2f} (1000), {rapport_5000:.2f} (5000), "
    f"{rapport_25000:.2f} (25000). "
    + phrase_rapport +
    f"Avec la correction de population finie (facteur {correction:.2f}), le rapport devient "
    f"{rapport_corrige:.2f}."
)
print(pourquoi_la_formule_deborde)

# ---- Les contrôles du sujet (ne pas modifier) ----
controle('vingt tirages de 25 000 lignes', lambda: len(ecarts_25000) == 20,
         'la taille demandée est-elle compatible avec celle du jeu de contrôle ?')
controle('la barre observée et la formule ont été chiffrées séparément',
         lambda: 0 < ecart_type_25000 < 10 and 0 < formule_25000 < 10,
         'deux nombres : la mesure observée, puis la formule — pas l un recopié dans l autre')
controle('le rapport est un nombre positif', lambda: 0 < rapport_25000 < 10)
controle('l explication tient en une phrase complète',
         lambda: len(pourquoi_la_formule_deborde) > 60,
         'la formule suppose des tirages indépendants — et là ?')

### Question 7 — Le pas d'apprentissage : qu'est-ce qui change ?  ·  *carnet J1*  ·  *cœur*


Trois valeurs de `learning_rate` — **0,05 / 0,5 / 2** — le reste inchangé. `learning_rate` est le
**pas d'apprentissage** : de combien chaque arbre corrige l'erreur du précédent. C'est le seul
réglage que J1 §6 n'a pas montré (elle n'y a montré que `max_iter`, le nombre d'arbres *demandé*).
Et l'arrêt anticipé s'en mêle : le modèle peut s'arrêter **tout seul** quand sa part interne
cesse de progresser, si bien que le nombre d'arbres *construits* peut différer de celui qu'on
a demandé.

À faire :
  0. **AVANT de mesurer** : `pred_pas` — ce que vous attendez des trois scores et du nombre
     d'arbres construits, avec la raison ;

Puis, dans UN SEUL tableau (`reglages`), une ligne par valeur :
  1. le score de validation, en pourcentage ;
  2. le nombre d'arbres **effectivement construits** (`m.n_iter_`) — pas celui demandé ;
  3. l'écart à la référence, **divisé par la barre de bruit** : combien de barres ?

Puis deux phrases, et elles sont le cœur de la question :
  · `score_separe` : les trois scores sont-ils séparés de **plus de deux barres** ?
  · `ce_qui_borne` : qu'est-ce qui borne **vraiment** ce modèle — le pas, ou autre chose ?
    Appuyez-vous sur ce que vous avez observé, `n_iter_` compris, et dites si cela confirme
    votre `pred_pas`.

À écrire : `pred_pas`, le tableau, `score_separe`, `ce_qui_borne`.


In [ ]:
# TODO : votre prédiction, AVANT toute mesure.
# pred_pas = … (à vous : les trois scores, les arbres construits, et pourquoi)
# TODO : un seul tableau, une ligne par learning_rate (0,05 / 0,5 / 2).
# reglages = … (à vous : score, arbres construits, écart à la référence en barres)
# score_separe = … (à vous)
# ce_qui_borne = … (à vous)

controle('la prédiction est écrite avant le tableau', lambda: len(pred_pas) > 40,
         'ce que vous attendez des scores et des arbres construits, et pourquoi')
controle('trois lignes, une par pas d apprentissage', lambda: len(reglages) == 3)
controle('chaque ligne porte un score en pourcentage',
         lambda: all(50 < r['score'] <= 100 for r in reglages),
         'un score de validation, en pourcentage')
controle('chaque ligne porte son NOMBRE D ARBRES CONSTRUITS',
         lambda: all(1 <= r['arbres'] <= 200 for r in reglages),
         'm.n_iter_ — le nombre d arbres réellement construits')
controle('chaque ligne rapporte son écart en barres',
         lambda: all(abs(r['en_barres']) < 20 for r in reglages),
         'l écart à la référence, divisé par la barre')
controle('les deux phrases sont écrites',
         lambda: len(score_separe) > 40 and len(ce_qui_borne) > 40,
         'deux phrases, appuyées sur votre tableau')


In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier as HGB

# ---- Prédiction, écrite AVANT toute mesure ----
pred_pas = (
    "J'attends des trois scores très proches (dans le bruit) : le signal est limité par les "
    "données et non par le pas. Avec 2, le modèle risque d'être instable ; avec 0,05 il "
    "apprend lentement. L'arrêt anticipé devrait arrêter 0,5 et 2 plus tôt que 0,05 "
    "(moins d'arbres construits)."
)

# ---- Mesures : un modèle par pas d'apprentissage ----
pas = (0.05, 0.5, 2)
modeles7 = [HGB(learning_rate=lr, random_state=0).fit(Xtr, ytr) for lr in pas]
scores7 = [100 * m.score(Xval, yval) for m in modeles7]
arbres7 = [int(m.n_iter_) for m in modeles7]

reglages = [{"learning_rate": lr, "score": s, "arbres": a, "en_barres": (s - reference) / erreur_type}
            for lr, s, a in zip(pas, scores7, arbres7)]
display(pd.DataFrame(reglages))

# ---- Les deux phrases (elles suivent vos mesures) ----
liste_scores = " / ".join(f"{s:.2f}" for s in scores7)
ecart_max_barres = (max(scores7) - min(scores7)) / erreur_type
separes = ecart_max_barres >= 2
bat_reference = all(r["en_barres"] >= 2 for r in reglages)
plus_tot = arbres7[1] <= arbres7[0] and arbres7[2] <= arbres7[0]

score_separe = (
    f"Scores pour 0,05 / 0,5 / 2 : {liste_scores} %. Écart maximal entre eux : "
    f"{ecart_max_barres:.2f} barre(s). "
    + ("Plus de 2 barres : les trois scores sont séparés."
       if separes else
       "Moins de 2 barres : les trois scores ne sont pas séparés, ce qui les distingue est dans le bruit.")
)

ce_qui_borne = (
    f"Arbres construits pour 0,05 / 0,5 / 2 : {arbres7}. "
    + ("Le pas ne change pas le résultat : ce qui borne le modèle est ailleurs, dans l'information contenue dans les colonnes. Cela confirme ma prédiction de scores proches. "
       if not separes else
       "Le pas a un effet mesurable sur le score, ce qui contredit ma prédiction de scores proches. ")
    + ("Les trois dépassent la référence de plus de 2 barres. "
       if bat_reference else
       "Au moins un des trois ne dépasse pas la référence de 2 barres. ")
    + ("L'arrêt anticipé a coupé plus tôt les grands pas, comme prévu."
       if plus_tot else
       "Le nombre d'arbres ne suit pas ma prédiction : l'arrêt anticipé a joué autrement que prévu.")
)
print(score_separe)
print(ce_qui_borne)

# ---- Les contrôles du sujet (ne pas modifier) ----
controle('la prédiction est écrite avant le tableau', lambda: len(pred_pas) > 40,
         'ce que vous attendez des scores et des arbres construits, et pourquoi')
controle('trois lignes, une par pas d apprentissage', lambda: len(reglages) == 3)
controle('chaque ligne porte un score en pourcentage',
         lambda: all(50 < r['score'] <= 100 for r in reglages),
         'un score de validation, en pourcentage')
controle('chaque ligne porte son NOMBRE D ARBRES CONSTRUITS',
         lambda: all(1 <= r['arbres'] <= 200 for r in reglages),
         'm.n_iter_ — le nombre d arbres réellement construits')
controle('chaque ligne rapporte son écart en barres',
         lambda: all(abs(r['en_barres']) < 20 for r in reglages),
         'l écart à la référence, divisé par la barre')
controle('les deux phrases sont écrites',
         lambda: len(score_separe) > 40 and len(ce_qui_borne) > 40,
         'deux phrases, appuyées sur votre tableau')

### Question 8 — Trois essais de trois natures — et le meilleur de N  ·  *carnet J1*  ·  *cœur*


Au moins **trois essais à vous**, et pas trois fois la même idée. La règle de l'ordre est
**appliquée** : `noter(...)` refuse un essai que vous n'avez pas annoncé.

À faire :
  1. trois essais de **natures différentes** : une **donnée** (une colonne, une correction), un
     **réglage** (un paramètre du modèle), une **famille de modèle** (un autre algorithme) ;
  2. `prevoir(...)` d'abord, puis la mesure dans `essai(...)` — il date la mesure lui-même ;
  3. remplissez `mes_essais` (une **table**, pas une liste) : une ligne par essai, avec **l'essai,
     sa nature, l'attendu, le mesuré, le gain en barres, et le verdict** — « réel » ou « dans le
     bruit ». Le gain se compte **par rapport au modèle de la mise en place** (`score_interne`), pas
     par rapport à la référence : c'est le *changement* qui décide du verdict — et un changement nul
     est une réponse, pas un échec ;
  4. `optimisme` : votre meilleur essai a été choisi **parce qu'il gagne sur cette validation**.
     Écrivez ce que ce choix fait à votre estimation du gain — une phrase, **avant** de mesurer
     (`optimisme`) — puis refaites la mesure du même essai sur un **second découpage** (une autre
     graine) : `verifie_par_second_tirage`, le gain de ce tirage, en barres.

À écrire : `mes_essais`, `optimisme`, `verifie_par_second_tirage`.


In [ ]:
# TODO : trois fois prevoir(...), puis essai(...) ; ensuite le tableau.
# mes_essais = … (à vous : essai, nature, attendu, mesuré, gain en barres, verdict)
# optimisme = … (à vous : votre phrase, écrite avant le second tirage)
# verifie_par_second_tirage = … (à vous : le gain sur un autre découpage, en barres)

controle('trois essais À VOUS (les amorces ne comptent pas)',
         lambda: sum(1 for e in ESSAIS
                     if e['hypothèse'] not in AMORCES) >= 3,
         'annoncez avec prevoir(...) : noter(...) refuse un essai non annoncé',
         des_que=lambda: any(e['hypothèse'] not in AMORCES for e in ESSAIS))
controle('trois natures DIFFÉRENTES',
         lambda: len({str(r['nature']).strip().lower() for r in mes_essais.to_dict('records')}) >= 3,
         'une donnée, un réglage, une famille de modèle')
controle('chaque ligne porte un gain EN BARRES',
         lambda: all(abs(r['gain_en_barres']) < 20 for r in mes_essais.to_dict('records')),
         'le changement divisé par la barre de bruit')
controle('chaque verdict est « réel » ou « dans le bruit »',
         lambda: all(str(r['verdict']).strip().lower() in ('réel', 'reel', 'dans le bruit')
                     for r in mes_essais.to_dict('records')),
         'deux valeurs admises, et rien d autre')
controle('la réponse est écrite avant le second tirage',
         lambda: len(optimisme) > 40,
         'une phrase, écrite AVANT de refaire la mesure')
controle('le second tirage est chiffré en barres',
         lambda: abs(verifie_par_second_tirage) < 20,
         'le même essai, une autre graine : le gain, en barres')


In [ ]:
import numpy as np
from sklearn.ensemble import HistGradientBoostingClassifier as HGB, RandomForestClassifier
from sklearn.model_selection import train_test_split

E1 = "âge corrigé (donnée)"
E2 = "max_leaf_nodes=15 (réglage)"
E3 = "forêt aléatoire (famille)"
natures = {E1: "donnée", E2: "réglage", E3: "famille de modèle"}

# ---- Les tables et les modèles de chaque cas (« base » = modèle de la mise en place) ----
Xage = X_sans_features.copy()
Xage["age"] = Xage["age"].where(Xage["age"] >= 0)
Xforet = pd.get_dummies(X_sans_features, columns=CAT).fillna(0)

cas = {
    E1: (Xage, lambda: HGB(random_state=0)),
    E2: (X_sans_features, lambda: HGB(max_leaf_nodes=15, random_state=0)),
    E3: (Xforet, lambda: RandomForestClassifier(n_estimators=100, min_samples_leaf=5, n_jobs=-1, random_state=0)),
    "base": (X_sans_features, lambda: HGB(random_state=0)),
}

# score (en fraction) d'un cas, pour un découpage donné par sa graine
def score_sur(nom, graine): itr, iva = train_test_split(X_sans_features.index, test_size=0.2, stratify=y, random_state=graine); Xc, fabrique = cas[nom]; m = fabrique(); m.fit(Xc.loc[itr], y.loc[itr]); return m.score(Xc.loc[iva], y.loc[iva])

# ---- Prédictions, AVANT toute mesure ----
prevoir(E1, "léger +", "-1 est un faux âge ; l'arbre le contourne déjà, gain faible")
prevoir(E2, "~0", "moins de feuilles : modèle plus simple, signal inchangé")
prevoir(E3, "-", "la forêt est en général moins fine que le boosting sur ces données")

# ---- Mesures (essai date la mesure lui-même) ----
scores = {nom: essai(nom, lambda nom=nom: score_sur(nom, 0), changement=natures[nom]) for nom in (E1, E2, E3)}

# ---- Le tableau des essais ----
attendus = {E1: "léger +", E2: "~0", E3: "-"}
gains = {nom: (s - score_interne) / erreur_type for nom, s in scores.items()}
mes_essais = pd.DataFrame([
    {"essai": nom, "nature": natures[nom], "attendu": attendus[nom], "mesuré": round(scores[nom], 2),
     "gain_en_barres": round(gains[nom], 2), "verdict": "réel" if abs(gains[nom]) >= 2 else "dans le bruit"}
    for nom in scores])
display(mes_essais)

# ---- Écrit AVANT le second tirage ----
optimisme = (
    "J'ai choisi le meilleur essai parce qu'il gagne sur CETTE validation : son gain est donc "
    "surestimé (le hasard du découpage a aidé), et il devrait en partie fondre sur un autre "
    "découpage."
)

# ---- Second découpage (graine 1) : le même essai, et le modèle de base sur ce même découpage ----
meilleur = max(scores, key=scores.get)
gain2 = 100 * (score_sur(meilleur, 1) - score_sur("base", 1)) / erreur_type
verifie_par_second_tirage = float(gain2)
print(meilleur, "| gain sur le second découpage (barres) :", round(gain2, 2))

# ---- Les contrôles du sujet (ne pas modifier) ----
controle('trois essais À VOUS (les amorces ne comptent pas)',
         lambda: sum(1 for e in ESSAIS
                     if e['hypothèse'] not in AMORCES) >= 3,
         'annoncez avec prevoir(...) : noter(...) refuse un essai non annoncé',
         des_que=lambda: any(e['hypothèse'] not in AMORCES for e in ESSAIS))
controle('trois natures DIFFÉRENTES',
         lambda: len({str(r['nature']).strip().lower() for r in mes_essais.to_dict('records')}) >= 3,
         'une donnée, un réglage, une famille de modèle')
controle('chaque ligne porte un gain EN BARRES',
         lambda: all(abs(r['gain_en_barres']) < 20 for r in mes_essais.to_dict('records')),
         'le changement divisé par la barre de bruit')
controle('chaque verdict est « réel » ou « dans le bruit »',
         lambda: all(str(r['verdict']).strip().lower() in ('réel', 'reel', 'dans le bruit')
                     for r in mes_essais.to_dict('records')),
         'deux valeurs admises, et rien d autre')
controle('la réponse est écrite avant le second tirage',
         lambda: len(optimisme) > 40,
         'une phrase, écrite AVANT de refaire la mesure')
controle('le second tirage est chiffré en barres',
         lambda: abs(verifie_par_second_tirage) < 20,
         'le même essai, une autre graine : le gain, en barres')

### Question 9 — Ce que le linéaire coûte, en barres — et ce qu'un coefficient ne dit pas  ·  *carnet J2*  ·  *cœur*


**La règle de lecture, posée ici** : un écart inférieur à **deux barres de bruit** n'est pas
une différence. Divisez, et dites combien de barres.

À faire :
  1. l'écart entre les deux familles est mesuré par la mise en place (`score_interne` et
     `score_lineaire`) : `rapport` = cet écart **divisé par la barre**. Est-ce une différence ?
  2. **des coefficients comparables entre eux** : les colonnes n'ont pas les mêmes unités, donc
     la taille brute d'un coefficient ne veut rien dire d'une colonne à l'autre. Deux routes :
     prendre des **indicatrices** (`quartier_…`, `assurance_…` — J2 §1 les construit), ou
     **centrer-réduire** vous-même les colonnes que vous lisez — `(x - x.mean()) / x.std()`, la mise
     à l'échelle que J2 §1 annonce **souhaitable mais non faite**. Nommez les trois colonnes ;
  3. `age` et `age_mois` disent la même chose, `montant_acte` et `cout_acte` aussi : ne les
     utilisez pas pour lire un coefficient, et dites pourquoi (`pourquoi_pas_les_dupliquees`) ;
  4. `quartier_provoque` : le coefficient d'un quartier dit-il que ce quartier **provoque** plus
     d'absences ? **Quelle autre lecture** en est possible ? Une phrase, pas trois ;
  5. `cout_et_rapport` : ce que le linéaire coûte (indicatrices, comblement, mise à
     l'échelle) et ce qu'il rapporte (des coefficients signés, un par colonne).

À écrire : `rapport`, les trois coefficients, `pourquoi_pas_les_dupliquees`, `quartier_provoque`,
`cout_et_rapport`.


In [ ]:
# TODO : l'écart des deux familles, divisé par la barre de bruit.
# rapport = … (à vous)
# TODO : trois coefficients COMPARABLES, lus sur des colonnes de même nature ou remises à
#        la même échelle ; nommez-les.
# trois_coefficients = … (à vous)
# pourquoi_pas_les_dupliquees = … (à vous)
# quartier_provoque = … (à vous : cause, ou autre chose ?)
# cout_et_rapport = … (à vous : ce qu il coûte, ce qu il rapporte)

controle('le rapport écart / barre est un nombre fini',
         lambda: isinstance(rapport, (int, float)) and abs(rapport) < 1e3)
controle('trois coefficients nommés', lambda: len(trois_coefficients) == 3,
         'trois noms de colonnes COMPARABLES entre elles')
controle('aucun coefficient choisi n est une colonne dupliquée',
         lambda: not ({str(c).split('_')[0] for c in trois_coefficients}
                      & {'age', 'montant', 'cout', 'dossier', 'patient'}),
         'age/age_mois et montant_acte/cout_acte disent la même chose')
controle('le refus des dupliquées est argumenté',
         lambda: len(pourquoi_pas_les_dupliquees) > 60)
controle('la lecture du quartier distingue cause et corrélation',
         lambda: len(quartier_provoque) > 60,
         'un coefficient décrit une association, sur des données observées')
controle('le coût et le rapport sont écrits', lambda: len(cout_et_rapport) > 60)


In [ ]:
# ---- 1. L'écart arbres / linéaire, en barres de bruit ----
rapport = float((score_interne - score_lineaire) / erreur_type)
print("écart arbres/linéaire en barres :", round(rapport, 2))

# ---- 2. Trois coefficients comparables : des indicatrices (0 ou 1, même échelle) ----
coefs = pd.Series(modele_lineaire.coef_[0], index=Xtr_n.columns)
base = [c for c in coefs.index if c.startswith(("quartier_", "assurance_"))]
gros = [c for c in base if Xtr_n[c].sum() >= 500]
choisis = gros if len(gros) >= 3 else base
trois_coefficients = list(coefs[choisis].abs().sort_values(ascending=False).index[:3])
print(coefs[trois_coefficients])

# ---- 3. Les phrases ----
pourquoi_pas_les_dupliquees = (
    "age et age_mois (et montant_acte et cout_acte) portent la même information : le modèle "
    "répartit le poids entre elles de façon arbitraire, donc chaque coefficient pris seul n'a "
    "plus de sens, et il dépend aussi de l'unité."
)
quartier_provoque = (
    "Non : un coefficient décrit une association sur des données observées, pas une cause. "
    "Autre lecture : le quartier peut refléter la distance, le revenu ou l'accès aux transports "
    "(variable cachée), pas le quartier lui-même."
)
cout_et_rapport = (
    "Coût : indicatrices (beaucoup de colonnes), comblement des valeurs manquantes (fillna) et "
    "mise à l'échelle à faire. Rapport : un coefficient signé par colonne, lisible, que "
    "l'on peut expliquer à un non-spécialiste, pour un écart de score de "
    f"{rapport:.1f} barre(s)."
)

# ---- Les contrôles du sujet (ne pas modifier) ----
controle('le rapport écart / barre est un nombre fini',
         lambda: isinstance(rapport, (int, float)) and abs(rapport) < 1e3)
controle('trois coefficients nommés', lambda: len(trois_coefficients) == 3,
         'trois noms de colonnes COMPARABLES entre elles')
controle('aucun coefficient choisi n est une colonne dupliquée',
         lambda: not ({str(c).split('_')[0] for c in trois_coefficients}
                      & {'age', 'montant', 'cout', 'dossier', 'patient'}),
         'age/age_mois et montant_acte/cout_acte disent la même chose')
controle('le refus des dupliquées est argumenté',
         lambda: len(pourquoi_pas_les_dupliquees) > 60)
controle('la lecture du quartier distingue cause et corrélation',
         lambda: len(quartier_provoque) > 60,
         'un coefficient décrit une association, sur des données observées')
controle('le coût et le rapport sont écrits', lambda: len(cout_et_rapport) > 60)

### Question 10 — Votre feature — différente de celle du carnet, et son contournement  ·  *carnet J2*  ·  *approfondissement*


**Écrivez une feature que le carnet n'a pas écrite.** `delai_long` est l'exemple de la séance 2,
et la recopier n'apprend rien : votre feature doit découler d'**un de vos constats de la
question 3**, et vous devez dire lequel.

À faire :
  1. dans `ajouter_features` — **la seule cellule de la mise en place faite pour être modifiée** —
     écrivez votre feature, et notez `constat_d_origine` (l'observation du constat qui l'a
     motivée, telle qu'elle est écrite dans le tableau) ;
  2. `prevoir(...)`, puis `remesurer()` : la chaîne passe par `preparer()`, donc `X` reçoit votre
     feature **une seule fois**, et `modele` est mis à jour ;
  3. `gain_de_ma_feature`, en barres de bruit — **un gain de zéro est une réponse** : la table est
     sur un plateau, et une feature raisonnable peut n'y rien changer ;
  4. **contournez `preparer()`** : appelez `modele.predict` sur une table préparée à la main en
     oubliant une étape → `score_contourne`, `ecart_du_contournement` (en barres), et
     `quelle_etape_oubliee`.

À écrire : la feature, son constat d'origine, le gain, le contournement et ce qu'il a produit.


In [ ]:
# TODO : votre feature, dans `ajouter_features` (mise en place) — PAS `delai_long`.
# et ici : le constat qui la motive, le gain, le contournement.
# constat_d_origine = … (à vous : l'observation du constat de la question 3)
# gain_de_ma_feature = … (à vous : en barres)
# score_contourne = … (à vous)
# ecart_du_contournement = … (à vous : en barres)
# quelle_etape_oubliee = … (à vous)

controle('la feature s inspire d un constat de la question 3',
         lambda: constat_d_origine in [c['observation'] for c in CONSTATS],
         'recopiez l observation exacte du constat qui motive votre feature')
controle('votre feature a enrichi la table', lambda: X.shape[1] > n_colonnes_avant,
         'elle entre par la chaîne, pas à la main',
         des_que=lambda: not ajouter_features(X_sans_features).equals(X_sans_features))
controle('le gain est chiffré en barres', lambda: abs(gain_de_ma_feature) < 20)
controle('l écart du contournement est chiffré en barres',
         lambda: abs(ecart_du_contournement) < 20,
         'un modèle entraîné sur une table et interrogé sur une autre')
controle('l étape oubliée est nommée', lambda: len(quelle_etape_oubliee) > 20,
         'nommez l étape manquante : c est elle, le bug silencieux')


In [ ]:
import numpy as np

# ---- 0. La feature : un drapeau « âge impossible » + l'âge corrigé (le -1 devient une valeur manquante) ----
def ajouter_features(df): df = df.copy(); df["age_manquant"] = (df["age"] < 0).astype(int); df["age_corrige"] = df["age"].where(df["age"] >= 0); return df

# nombre de colonnes AVANT la feature (la table nue)
n_colonnes_avant = X_sans_features.shape[1]

# ---- 1. Le constat d'origine : retrouvé dans le tableau de la question 3 ----
constat_d_origine = [c["observation"] for c in CONSTATS if "âge impossible" in c["observation"]][0]

# ---- 2. Prédiction AVANT la mesure, puis mesure ----
prevoir("feature âge corrigé", "~0 ou léger +",
        "l'arbre gérait déjà le -1 : la table est sur un plateau")
score_avec_feature = remesurer()
gain_de_ma_feature = float((score_avec_feature - score_interne) / erreur_type)
print("score avec la feature :", round(score_avec_feature, 2), "| gain en barres :", round(gain_de_ma_feature, 2))

# ---- 3. Le contournement : table préparée À LA MAIN, en oubliant le fillna("") des catégories ----
brut = train.loc[Xval.index]
table_main = brut.drop(columns=[c for c in A_ECARTER + TROP_FIN + [CIBLE] if c in brut.columns]).copy()
for c in CAT: table_main[c] = table_main[c].astype("category").cat.set_categories(X[c].cat.categories)
table_main = ajouter_features(table_main)

score_chaine = 100 * (predict(brut) == yval.values).mean()
score_contourne = 100 * (modele.predict(table_main) == yval.values).mean()
ecart_du_contournement = float((score_contourne - score_chaine) / erreur_type)
n_nan_cat = int(table_main[CAT].isna().sum().sum())
print("par la chaîne :", round(score_chaine, 2), "| contourné :", round(score_contourne, 2), "| écart en barres :", round(ecart_du_contournement, 2))
print("valeurs manquantes dans les colonnes catégorielles :", n_nan_cat)

quelle_etape_oubliee = (
    "L'étape oubliée est le comblement des valeurs manquantes des colonnes catégorielles par la "
    "chaîne vide (fillna('')). "
    f"Il y a {n_nan_cat} valeur(s) manquante(s) dans ces colonnes sur la validation. "
    + ("Sans cette étape, elles ne tombent pas dans la même catégorie qu'à l'entraînement : "
       "aucune erreur n'est levée, le score bouge en silence. C'est un bug silencieux."
       if n_nan_cat > 0 else
       "Ici aucune n'est manquante, donc l'oubli ne change rien sur ces données : le bug existe "
       "mais reste invisible, et c'est justement le danger d'un bug silencieux.")
)
print(quelle_etape_oubliee)

# ---- Les contrôles du sujet (ne pas modifier) ----
controle('la feature s inspire d un constat de la question 3',
         lambda: constat_d_origine in [c['observation'] for c in CONSTATS],
         'recopiez l observation exacte du constat qui motive votre feature')
controle('votre feature a enrichi la table', lambda: X.shape[1] > n_colonnes_avant,
         'elle entre par la chaîne, pas à la main',
         des_que=lambda: not ajouter_features(X_sans_features).equals(X_sans_features))
controle('le gain est chiffré en barres', lambda: abs(gain_de_ma_feature) < 20)
controle('l écart du contournement est chiffré en barres',
         lambda: abs(ecart_du_contournement) < 20,
         'un modèle entraîné sur une table et interrogé sur une autre')
controle('l étape oubliée est nommée', lambda: len(quelle_etape_oubliee) > 20,
         'nommez l étape manquante : c est elle, le bug silencieux')

### Question 11 — La colonne qui disparaît : la trouver, mesurer son gain, et le voir s'évaporer  ·  *carnet J2*  ·  *cœur*


**Cherchez-la.** J1 l'a rangée dans `A_ECARTER` — parmi dix autres, et sans dire pourquoi.
Ce qui la trahit est un **écart de remplissage** entre `train` et `test`.

Le gain et la simulation se mesurent sur **`mon_train`** — votre échantillon, comme à la
question 2 : un gain recopié d'un voisin ne vaudra pas le vôtre.

À faire :
  1. calculez, **colonne par colonne**, l'écart de remplissage entre `train` et `test`, triez, et
     nommez celle qui se détache (`colonne_trouvee`) avec son écart (`ecart_remplissage`, en
     points). La **deuxième** du tri mérite la même attention : dites pour elle aussi pourquoi elle
     n'est pas la réponse (`pourquoi_pas_la_derivee`) ;
  2. `remplissage_du_test` : quelle **part du test** porte encore une valeur dans cette colonne ?
  3. `gain_interne` : le même modèle, avec et sans cette colonne, sur la **même** validation —
     le gain en points, et en barres (`gain_en_barres`) ;
  4. **la simulation, et c'est le cœur** : entraînez avec la colonne, puis rendez-la manquante
     dans la validation **pour une part de lignes égale à celle du test** (le taux de trous du
     test). `score_simule`, `chute_en_barres` : que reste-t-il du gain ?
  5. `decision_A_ECARTER` : l'écarter était-il la bonne décision ? Était-ce pour la bonne
     raison ? Une phrase pour chacune des deux questions.

À écrire : la colonne, l'écart, la part du test, le gain interne, la simulation, la décision.


In [ ]:
# TODO : l'écart de remplissage entre train et test, colonne par colonne.
# ecarts = … (à vous)
# colonne_trouvee = … (à vous)
# ecart_remplissage = … (à vous : en points)
# pourquoi_pas_la_derivee = … (à vous)
# remplissage_du_test = … (à vous : en pourcentage)
# gain_interne = … (à vous : en points)
# gain_en_barres = … (à vous)
# TODO : la simulation — la colonne manquante au taux du test, dans la validation.
# score_simule = … (à vous)
# chute_en_barres = … (à vous)
# decision_A_ECARTER = … (à vous)

controle('l écart de remplissage est un pourcentage', lambda: 0 <= ecart_remplissage <= 100)
controle('la colonne nommée est bien celle de VOTRE tri',
         lambda: colonne_trouvee == ecarts.index[0],
         'la première de votre propre tri : l écart est net, il n y a pas d hésitation')
controle('la colonne qui dérive est écartée, avec sa raison',
         lambda: len(pourquoi_pas_la_derivee) > 40,
         'une dérive n est pas une disparition : chiffrez-la')
controle('la part du test encore renseignée est un pourcentage',
         lambda: 0 <= remplissage_du_test <= 100,
         'ce qui survit dans le test : un pourcentage, la part encore renseignée')
controle('le gain interne et sa simulation sont chiffrés',
         lambda: abs(gain_en_barres) < 40 and abs(chute_en_barres) < 60,
         'deux nombres, chacun divisé par la barre')
controle('la simulation a été mesurée sur un score à part',
         lambda: 0 < abs(chute_en_barres) < 60,
         'le même modèle, la validation rendue trouée : comparez les deux scores')
controle('la décision est argumentée, sur les deux questions',
         lambda: len(decision_A_ECARTER) > 80,
         'la bonne décision, et la bonne raison')


In [ ]:
import numpy as np
from sklearn.ensemble import HistGradientBoostingClassifier as HGB
from sklearn.model_selection import train_test_split

# ---- 1. L'écart de remplissage entre train et test, colonne par colonne ----
communes = [c for c in train.columns if c in test.columns]
ecarts = ((train[communes].notna().mean() - test[communes].notna().mean()).abs() * 100).sort_values(ascending=False)
print(ecarts.head(5))

colonne_trouvee = ecarts.index[0]
ecart_remplissage = float(ecarts.iloc[0])
seconde = ecarts.index[1]
remplissage_seconde = 100 * test[seconde].notna().mean()
pourquoi_pas_la_derivee = (
    f"{seconde} : écart de {ecarts.iloc[1]:.1f} points, renseignée à {remplissage_seconde:.1f} % dans le test "
    f"contre {100 * train[seconde].notna().mean():.1f} % dans train. "
    + ("Elle reste renseignée dans la plupart des lignes du test : c'est une dérive, pas une disparition ; on la surveille, on ne l'écarte pas pour cette raison."
       if remplissage_seconde > 50 else
       "Elle est elle aussi presque vide dans le test : elle se traite comme la première.")
)
remplissage_du_test = float(100 * test[colonne_trouvee].notna().mean())
print("colonne trouvée :", colonne_trouvee, "| écart :", round(ecart_remplissage, 1), "| remplie dans le test :", round(remplissage_du_test, 1), "%")

# ---- 2. Gain interne, sur mon_train, même validation avec et sans la colonne ----
col = colonne_trouvee
ym = mon_train[CIBLE]
Xm_tout, _, _ = matrice(mon_train)
Xm_sans = Xm_tout.drop(columns=[col], errors="ignore")
Xm_avec = Xm_sans.copy()
Xm_avec[col] = mon_train[col]
col_num = pd.api.types.is_numeric_dtype(Xm_avec[col])
Xm_avec[col] = Xm_avec[col] if col_num else (Xm_avec[col].astype("category") if Xm_avec[col].nunique() <= 255 else Xm_avec[col].astype("category").cat.codes.replace(-1, np.nan))

itr, iva = train_test_split(Xm_avec.index, test_size=0.2, stratify=ym, random_state=0)
a1, b1, a0, b0 = Xm_avec.loc[itr], Xm_avec.loc[iva], Xm_sans.loc[itr], Xm_sans.loc[iva]
c_tr, d_va = ym.loc[itr], ym.loc[iva]

m_avec = HGB(random_state=0).fit(a1, c_tr)
m_sans = HGB(random_state=0).fit(a0, c_tr)
s_avec_col = 100 * m_avec.score(b1, d_va)
s_sans_col = 100 * m_sans.score(b0, d_va)
barre_m = 100 * ((s_avec_col / 100) * (1 - s_avec_col / 100) / len(d_va)) ** 0.5
gain_interne = s_avec_col - s_sans_col
gain_en_barres = gain_interne / barre_m

# ---- 3. Simulation : la colonne devient manquante dans la validation, au taux de trous du test ----
taux_trous = 1 - remplissage_du_test / 100
b_sim = b1.copy()
b_sim[col] = b_sim[col].astype(float) if pd.api.types.is_numeric_dtype(b_sim[col]) else b_sim[col]
masque = np.random.default_rng(0).random(len(b_sim)) < taux_trous
b_sim.loc[masque, col] = np.nan
score_simule = 100 * m_avec.score(b_sim, d_va)
chute_en_barres = (s_avec_col - score_simule) / barre_m
print("avec :", round(s_avec_col, 2), "| sans :", round(s_sans_col, 2), "| gain :", round(gain_interne, 2), "pt =", round(gain_en_barres, 2), "barres")
print("simulé :", round(score_simule, 2), "| chute :", round(chute_en_barres, 2), "barres")

# ---- 4. La décision ----
s_evapore = (gain_en_barres >= 2) and (chute_en_barres >= 0.5 * gain_en_barres)
decision_A_ECARTER = (
    f"Bonne décision : la colonne n'est renseignée que pour {remplissage_du_test:.1f} % des lignes du test. "
    + (f"Le gain interne ({gain_en_barres:.1f} barres) s'évapore dans la simulation (chute de {chute_en_barres:.1f} barres). "
       if s_evapore else
       f"Le gain interne est de {gain_en_barres:.1f} barres et la simulation fait chuter le score de {chute_en_barres:.1f} barres. ")
    + "Mais la bonne raison n'est pas le score : c'est la disparition de la colonne au moment de décider ; "
      "un gain mesuré en interne ne dit rien de ce qui se passera en production."
)
print(decision_A_ECARTER)

# ---- Les contrôles du sujet (ne pas modifier) ----
controle('l écart de remplissage est un pourcentage', lambda: 0 <= ecart_remplissage <= 100)
controle('la colonne nommée est bien celle de VOTRE tri',
         lambda: colonne_trouvee == ecarts.index[0],
         'la première de votre propre tri : l écart est net, il n y a pas d hésitation')
controle('la colonne qui dérive est écartée, avec sa raison',
         lambda: len(pourquoi_pas_la_derivee) > 40,
         'une dérive n est pas une disparition : chiffrez-la')
controle('la part du test encore renseignée est un pourcentage',
         lambda: 0 <= remplissage_du_test <= 100,
         'ce qui survit dans le test : un pourcentage, la part encore renseignée')
controle('le gain interne et sa simulation sont chiffrés',
         lambda: abs(gain_en_barres) < 40 and abs(chute_en_barres) < 60,
         'deux nombres, chacun divisé par la barre')
controle('la simulation a été mesurée sur un score à part',
         lambda: 0 < abs(chute_en_barres) < 60,
         'le même modèle, la validation rendue trouée : comparez les deux scores')
controle('la décision est argumentée, sur les deux questions',
         lambda: len(decision_A_ECARTER) > 80,
         'la bonne décision, et la bonne raison')

### Question 12 — Votre outil : `suspects(df, cible)`  ·  *carnet J2*  ·  *cœur*


Le carnet a montré **deux signes** d'une colonne qui connaît la réponse, mais il l'a fait une
colonne à la fois. Écrivez l'**outil** qui les cherche toutes — vous le réutiliserez en projet.

À faire :
  1. `suspects(df, cible)` rend une **table**, une ligne par colonne suspecte, avec au moins :
     le nom, son **écart de remplissage** entre les deux groupes de la cible, et le **signe**
     qui l'a trahie ;
     — **signe 1, le VIDE** : une colonne remplie d'un côté et vide de l'autre (écart de
       remplissage de plus de 50 points) ;
     — **signe 2, la VALEUR** : une colonne remplie **des deux côtés** dont une valeur ne vit
       que dans **un** groupe ;
     — votre outil doit **ignorer l'identifiant** et la cible elle-même ;
  2. appliquez-le à `train` : `table_suspects` ;
  3. Rendez `classement_suspects` — une table d'**une ligne par colonne signalée**, avec deux
     colonnes : `verdict` (à choisir dans **fuite** / **pas une fuite** / **à discuter**) et
     `justification` — et justifiez par **ce que la valeur veut dire**, jamais par le fait qu'elle
     est constante ;
  4. **le troisième signe, celui que les deux premiers ne voient pas** : une valeur
     **impossible** (une **sentinelle** : un code mis à la place d'une mesure manquante — le −1
     d'`age` en séance 1 en est une) portée par **peu** de lignes. `sentinelle_trouvee`
     (la colonne et la valeur), `lignes_porteuses` (combien de lignes), `taux_sur_ces_lignes`
     (le taux d'absence sur ces lignes, comparé au taux global), et `verdict_sentinelle` : fuite,
     ou défaut de saisie ?

À écrire : la fonction, `table_suspects`, `classement_suspects`, la sentinelle et son verdict.


In [ ]:
# À vous — écrivez votre réponse ici.

# TODO : l'outil, avec les DEUX signes, qui ignore identifiant et cible.
def suspects(df, cible):
    """Rend une table, une ligne par colonne suspecte : nom, écart de remplissage, signe.
       Signe 1 (le VIDE) : plus de 50 points d'écart de remplissage entre les deux groupes.
       Signe 2 (la VALEUR) : une valeur qui ne vit que dans un seul des deux groupes.
       Ajoutez ce que vous jugez utile — mais jamais en lisant la cible par accident."""
    # … (à vous)

# TODO : l'outil appliqué à `train`.
# table_suspects = … (à vous)
# classement_suspects = … (à vous)
# TODO : le troisième signe — la sentinelle.
# sentinelle_trouvee = … (à vous)
# lignes_porteuses = … (à vous)
# taux_sur_ces_lignes = … (à vous : en pourcentage)
# verdict_sentinelle = … (à vous)

controle('`suspects` rend une table, pas une liste de mots',
         lambda: hasattr(table_suspects, 'columns'),
         'une ligne par colonne suspecte : un nom, un écart de remplissage, un signe')
controle('l identifiant et la cible sont exclus',
         lambda: not ({ID, CIBLE} & set(table_suspects.iloc[:, 0])),
         'un identifiant sépare tout : il n apprend rien')
controle('le signe du VIDE apparaît au moins une fois',
         lambda: any('vide' in str(v).lower()
                     for v in table_suspects.to_numpy().ravel()),
         'une colonne remplie d un côté, vide de l autre')
controle('le signe de la VALEUR apparaît au moins une fois',
         lambda: any('valeur' in str(v).lower()
                     for v in table_suspects.to_numpy().ravel()),
         'une colonne remplie partout, dont une valeur ne vit que dans un groupe')
controle('chaque verdict a été choisi, et le tri n est pas uniforme',
         lambda: len(set(str(v).strip().lower() for v in classement_suspects['verdict'])) >= 2,
         'la même réponse recopiée sur toutes les lignes n est pas un tri')
controle('chaque colonne signalée est classée, et chaque verdict argumenté',
         lambda: len(classement_suspects) == len(table_suspects)
                 and all(len(str(x)) > 30 for x in classement_suspects['justification']),
         'ce que la valeur VEUT DIRE, pas le fait qu elle est constante')
controle('la sentinelle est nommée, chiffrée, et jugée',
         lambda: len(str(sentinelle_trouvee)) > 5 and 0 < lignes_porteuses < 5000
                 and 0 <= taux_sur_ces_lignes <= 100 and len(verdict_sentinelle) > 40,
         'une valeur impossible, sur peu de lignes : fuite, ou saisie ?')


In [ ]:
# ---- L'outil : deux petites aides, puis suspects ----
def _remplissage(df, cible): g1 = (df[cible] == 1); return pd.DataFrame({"r1": df[g1].notna().mean() * 100, "r0": df[~g1].notna().mean() * 100})

def _exclusives(df, cible, c): g1 = (df[cible] == 1); excl = set(df.loc[g1, c].dropna().unique()) ^ set(df.loc[~g1, c].dropna().unique()); return excl, int(df[c].isin(list(excl)).sum())

def suspects(df, cible): r = _remplissage(df, cible).drop(index=[cible, ID], errors="ignore"); r["ecart"] = (r["r1"] - r["r0"]).abs().round(1); lignes_vide = [{"colonne": c, "ecart_remplissage": r.loc[c, "ecart"], "signe": "vide"} for c in r.index if r.loc[c, "ecart"] > 50]; candidates = [c for c in r.index if r.loc[c, "ecart"] <= 50 and min(r.loc[c, "r1"], r.loc[c, "r0"]) > 0 and df[c].nunique() <= 50]; resultats = [(c,) + _exclusives(df, cible, c) for c in candidates]; lignes_valeur = [{"colonne": c, "ecart_remplissage": r.loc[c, "ecart"], "signe": "valeur exclusive à un groupe : " + str(sorted(map(str, e))[:3])} for c, e, n in resultats if e and n >= 30]; return pd.DataFrame(lignes_vide + lignes_valeur, columns=["colonne", "ecart_remplissage", "signe"])

# ---- Appliqué à train ----
table_suspects = suspects(train, CIBLE)
display(table_suspects)

# ---- Le classement : ce que chaque valeur VEUT DIRE ----
CONNUS = {
    "motif_absence": ("fuite", "Le motif n'existe que si le patient est absent : il est connu après l'issue, c'est la réponse elle-même."),
    "duree_consultation": ("fuite", "Une durée de consultation n'existe que si la consultation a eu lieu, donc si le patient est venu."),
    "constantes_relevees": ("fuite", "Les constantes sont relevées pendant la consultation : connues après l'issue, pas au moment de décider."),
}
defaut = lambda c: ("à discuter", f"Un signe est repéré sur {c}, mais il faut savoir si sa valeur est connue avant ou après l'issue du rendez-vous avant de conclure.")
classement_suspects = pd.DataFrame([{"colonne": c, "verdict": CONNUS.get(c, defaut(c))[0], "justification": CONNUS.get(c, defaut(c))[1]} for c in table_suspects["colonne"]], columns=["colonne", "verdict", "justification"])
display(classement_suspects)

# ---- Le troisième signe : la sentinelle ----
neg = train["age"] < 0
sentinelle_trouvee = "age = " + ", ".join(map(str, train.loc[neg, "age"].unique()))
lignes_porteuses = int(neg.sum())
taux_sur_ces_lignes = float(100 * train.loc[neg, CIBLE].mean())
taux_global = float(100 * train[CIBLE].mean())
barre_s = 100 * ((taux_global / 100) * (1 - taux_global / 100) / lignes_porteuses) ** 0.5
ecart_s = (taux_sur_ces_lignes - taux_global) / barre_s
print(sentinelle_trouvee, "|", lignes_porteuses, "lignes | taux :", round(taux_sur_ces_lignes, 1), "% | global :", round(taux_global, 1), "% |", round(ecart_s, 2), "barre(s)")

verdict_sentinelle = (
    f"age = -1 est un code mis à la place d'un âge manquant, porté par {lignes_porteuses} ligne(s) seulement ; "
    f"leur taux d'absence est de {taux_sur_ces_lignes:.1f} % contre {taux_global:.1f} % au global, soit {ecart_s:.1f} barre(s). "
    + ("L'écart est sous 2 barres : défaut de saisie, pas une fuite."
       if abs(ecart_s) < 2 else
       "L'écart dépasse 2 barres : on le surveille, car le fait d'être manquant porte peut-être de l'information, mais ce n'est pas la réponse elle-même.")
)
print(verdict_sentinelle)

# ---- Les contrôles du sujet (ne pas modifier) ----
controle('`suspects` rend une table, pas une liste de mots',
         lambda: hasattr(table_suspects, 'columns'),
         'une ligne par colonne suspecte : un nom, un écart de remplissage, un signe')
controle('l identifiant et la cible sont exclus',
         lambda: not ({ID, CIBLE} & set(table_suspects.iloc[:, 0])),
         'un identifiant sépare tout : il n apprend rien')
controle('le signe du VIDE apparaît au moins une fois',
         lambda: any('vide' in str(v).lower()
                     for v in table_suspects.to_numpy().ravel()),
         'une colonne remplie d un côté, vide de l autre')
controle('le signe de la VALEUR apparaît au moins une fois',
         lambda: any('valeur' in str(v).lower()
                     for v in table_suspects.to_numpy().ravel()),
         'une colonne remplie partout, dont une valeur ne vit que dans un groupe')
controle('chaque verdict a été choisi, et le tri n est pas uniforme',
         lambda: len(set(str(v).strip().lower() for v in classement_suspects['verdict'])) >= 2,
         'la même réponse recopiée sur toutes les lignes n est pas un tri')
controle('chaque colonne signalée est classée, et chaque verdict argumenté',
         lambda: len(classement_suspects) == len(table_suspects)
                 and all(len(str(x)) > 30 for x in classement_suspects['justification']),
         'ce que la valeur VEUT DIRE, pas le fait qu elle est constante')
controle('la sentinelle est nommée, chiffrée, et jugée',
         lambda: len(str(sentinelle_trouvee)) > 5 and 0 < lignes_porteuses < 5000
                 and 0 <= taux_sur_ces_lignes <= 100 and len(verdict_sentinelle) > 40,
         'une valeur impossible, sur peu de lignes : fuite, ou saisie ?')

### Question 13 — Le rappel : cause, ou autre chose ? — et l'expérience qui trancherait  ·  *carnet J2*  ·  *approfondissement*


Un rappel est envoyé, et l'absence est **plus** fréquente avec le rappel. Faut-il en conclure
que le rappel fait manquer les rendez-vous ?

**Tout se mesure sur `mon_train`** — votre échantillon, comme aux questions 2 et 11.

À faire :
  1. **AVANT de regarder les tranches** : `annonce` — le rappel aide-t-il, ou nuit-il, une fois la
     comparaison faite **à délai égal** ? Écrivez votre prédiction, et ce qui vous ferait changer
     d'avis ;
  2. le taux d'absence avec et sans rappel, **globalement** : `ecart_global` (en points) ;
  3. **à délai égal** : par tranches de délai, le taux avec et sans rappel dans **chaque** tranche,
     et pour chaque écart **sa barre de bruit** (`tranches`) — sans la barre, la conclusion
     contredit la règle du cours : un écart de moins de deux barres n'est pas un écart. Comptez
     les **effectifs** : toutes les tranches ne portent pas les deux groupes, la plus courte n'en
     porte qu'un — et sur **votre** échantillon, certaines tomberont **sous les 500 lignes** :
     celles-là ne se comparent pas, mettez-les dans `tranches_exclues` en disant pourquoi ;
  4. `variable_qui_explique` : la variable — s'il y en a une — qui explique l'écart global, avec
     le chiffre qui le montre (`delai_median_avec`, `delai_median_sans`) ; s'il n'y en a pas,
     dites-le ;
  5. `experience_qui_trancherait` : quelle expérience **trancherait vraiment** ? Décrivez-la en
     trois lignes — ce qu'on tirerait au hasard, ce qu'on comparerait, et pourquoi la comparaison
     serait alors honnête.

À écrire : l'annonce, l'écart global, les tranches avec leur barre, la variable, l'expérience.


In [ ]:
# TODO : votre annonce, AVANT de regarder les tranches.
# annonce = … (à vous)
# TODO : l'écart global, puis les tranches AVEC leur barre de bruit.
# ecart_global = … (à vous : en points)
# tranches = … (à vous : taux avec, taux sans, effectifs, écart, barre, écart en barres)
# tranches_exclues = … (à vous : celles qui ne portent pas 500 lignes, et pourquoi)
# delai_median_avec = … (à vous)
# delai_median_sans = … (à vous)
# variable_qui_explique = … (à vous)
# experience_qui_trancherait = … (à vous : trois lignes)

controle('l annonce est écrite avant les tranches', lambda: len(annonce) > 60,
         'ce que vous attendez, ET ce qui vous ferait changer d avis')
controle('l écart global est chiffré, en points', lambda: abs(ecart_global) < 100,
         'le taux avec rappel moins le taux sans rappel, en points')
controle('les tranches portent leur barre, pas seulement leur écart',
         lambda: len(tranches) >= 2 and all('barre' in r for r in tranches),
         'sans la barre, un écart de 2 points ne veut rien dire')
controle('les tranches trop fines de VOTRE échantillon sont mises à part',
         lambda: hasattr(tranches_exclues, '__len__')
                 and all(r.get('lignes', 0) >= 500 for r in tranches),
         'sous 500 lignes, on ne conclut rien : nommez-les au lieu de les comparer')
controle('chaque tranche comparée porte son écart et son effectif',
         lambda: all(abs(r['ecart']) > 0 and r.get('lignes', 0) > 0 for r in tranches),
         'un écart chiffré par tranche — et le nombre de lignes qui le porte')
controle('les effectifs sont comptés',
         lambda: all(r.get('lignes', 0) > 0 for r in tranches),
         'une tranche sans les deux groupes ne se compare pas')
controle('la variable qui explique est nommée, et les deux médianes sont chiffrées',
         lambda: len(str(variable_qui_explique)) > 3 and delai_median_avec >= 0
                 and delai_median_sans >= 0,
         'les deux médianes du délai, en jours, avec et sans rappel')
controle('l expérience qui trancherait est décrite en trois lignes',
         lambda: len(experience_qui_trancherait) > 120,
         'ce qu on tire au hasard, ce qu on compare, et pourquoi c est honnête')


In [ ]:
import numpy as np

# ---- Les deux colonnes utilisées (vérifiez les noms affichés) ----
RAPPEL = [c for c in mon_train.columns if "rappel" in c.lower()][0]
DELAI = [c for c in mon_train.columns if "delai" in c.lower() and pd.api.types.is_numeric_dtype(mon_train[c])][0]
print("colonnes utilisées :", RAPPEL, DELAI)

# ---- ÉCRITE AVANT de regarder les tranches ----
annonce = (
    "J'attends que le rappel ne nuise pas ou aide un peu une fois le délai égalisé : l'écart "
    "global vient d'une autre variable (le délai). Je changerais d'avis si, dans chaque tranche "
    "de délai, le taux d'absence avec rappel reste supérieur de plus de 2 barres."
)

# ---- L'écart global ----
avec = mon_train[RAPPEL].astype(str).str.lower().isin(["1", "1.0", "oui", "true", "yes"])
taux_avec = 100 * mon_train.loc[avec, CIBLE].mean()
taux_sans = 100 * mon_train.loc[~avec, CIBLE].mean()
ecart_global = float(taux_avec - taux_sans)
print("taux avec rappel :", round(taux_avec, 2), "| sans :", round(taux_sans, 2), "| écart global :", round(ecart_global, 2), "points")

# ---- À délai égal : une mesure par tranche de délai ----
bornes = [-1, 3, 7, 14, 30, 60, 90, 1e9]
tr = pd.cut(mon_train[DELAI], bornes)

def mesurer_tranche(t): m_t = (tr == t); n1 = int((m_t & avec).sum()); n0 = int((m_t & ~avec).sum()); p1 = mon_train.loc[m_t & avec, CIBLE].mean(); p0 = mon_train.loc[m_t & ~avec, CIBLE].mean(); barre = 100 * (p1 * (1 - p1) / max(n1, 1) + p0 * (1 - p0) / max(n0, 1)) ** 0.5; ecart = 100 * (p1 - p0); return {"tranche": str(t), "taux_avec": 100 * p1, "taux_sans": 100 * p0, "lignes": n1 + n0, "avec": n1, "sans": n0, "ecart": ecart, "barre": barre, "ecart_en_barres": (ecart / barre if barre > 0 else float("nan")), "ok": (n1 >= 500 and n0 >= 500)}

toutes = [mesurer_tranche(t) for t in tr.cat.categories]
tranches = [r for r in toutes if r["ok"]]
tranches_exclues = [{"tranche": r["tranche"], "avec": r["avec"], "sans": r["sans"], "pourquoi": "un des deux groupes a moins de 500 lignes"} for r in toutes if not r["ok"]]
display(pd.DataFrame(tranches))
display(pd.DataFrame(tranches_exclues))

# ---- La variable qui explique ----
delai_median_avec = float(mon_train.loc[avec, DELAI].median())
delai_median_sans = float(mon_train.loc[~avec, DELAI].median())
nb_signif = sum(1 for r in tranches if abs(r["ecart_en_barres"]) >= 2)
variable_qui_explique = (
    f"Le délai de prise de rendez-vous ({DELAI}) : médiane {delai_median_avec:.0f} jours avec rappel "
    f"contre {delai_median_sans:.0f} sans. "
    + ("Le rappel va surtout aux rendez-vous lointains, qui manquent plus souvent : le délai explique au moins une partie de l'écart global. "
       if delai_median_avec > delai_median_sans else
       "Le délai médian n'est pas plus long avec rappel : le délai n'explique pas l'écart global. ")
    + f"À délai égal, l'écart dépasse 2 barres dans {nb_signif} tranche(s) sur {len(tranches)}."
)
print(variable_qui_explique)

# ---- L'expérience qui trancherait ----
experience_qui_trancherait = (
    "Tirer AU HASARD, parmi les patients, ceux qui reçoivent un rappel et ceux qui n'en reçoivent pas. "
    "Comparer ensuite les taux d'absence des deux groupes. Le tirage au hasard rend les deux groupes "
    "semblables en délai et en tout le reste : la seule différence est le rappel, donc la comparaison est honnête."
)

# ---- Les contrôles du sujet (ne pas modifier) ----
controle('l annonce est écrite avant les tranches', lambda: len(annonce) > 60,
         'ce que vous attendez, ET ce qui vous ferait changer d avis')
controle('l écart global est chiffré, en points', lambda: abs(ecart_global) < 100,
         'le taux avec rappel moins le taux sans rappel, en points')
controle('les tranches portent leur barre, pas seulement leur écart',
         lambda: len(tranches) >= 2 and all('barre' in r for r in tranches),
         'sans la barre, un écart de 2 points ne veut rien dire')
controle('les tranches trop fines de VOTRE échantillon sont mises à part',
         lambda: hasattr(tranches_exclues, '__len__')
                 and all(r.get('lignes', 0) >= 500 for r in tranches),
         'sous 500 lignes, on ne conclut rien : nommez-les au lieu de les comparer')
controle('chaque tranche comparée porte son écart et son effectif',
         lambda: all(abs(r['ecart']) > 0 and r.get('lignes', 0) > 0 for r in tranches),
         'un écart chiffré par tranche — et le nombre de lignes qui le porte')
controle('les effectifs sont comptés',
         lambda: all(r.get('lignes', 0) > 0 for r in tranches),
         'une tranche sans les deux groupes ne se compare pas')
controle('la variable qui explique est nommée, et les deux médianes sont chiffrées',
         lambda: len(str(variable_qui_explique)) > 3 and delai_median_avec >= 0
                 and delai_median_sans >= 0,
         'les deux médianes du délai, en jours, avec et sans rappel')
controle('l expérience qui trancherait est décrite en trois lignes',
         lambda: len(experience_qui_trancherait) > 120,
         'ce qu on tire au hasard, ce qu on compare, et pourquoi c est honnête')

### Question 14 — Le rapport final — et une hypothèse testée sous son propre nom  ·  *carnet J1 + J2*  ·  *cœur*


Les trois tableaux se remplissent **dans les questions** ; cette question-ci produit les
hypothèses, et **en teste une**.

À faire :
  1. écrivez **trois** hypothèses de features avec `hypothese(...)` : ce qu'elle raconte (avec le
     **signe** attendu), comment on la teste, et ce qui la **contredirait** — un résultat précis,
     pas « si ça ne marche pas » ;
  2. **testez-en une** avec `essai(...)`, sous **le même nom** que l'hypothèse : la mesure, puis
     `confirmee_ou_contredite` — « confirmée » ou « contredite », jugée sur le critère que
     l'hypothèse avait annoncé **avant** la mesure ;
  3. un « contredite » n'est pas un échec : c'est le seul des deux verdicts qui apprend quelque
     chose. Écrivez ce que vous en faites (`suite_donnee`).

À écrire : les trois hypothèses, le test, le verdict, la suite donnée.


In [ ]:
# TODO : trois hypothèses, ici.
# hypothese("ma feature", "ce qu'elle raconte, avec le signe", "comment je la teste",
#           "ce qui la contredirait")
# TODO : une hypothèse TESTÉE, sous son propre nom.
# essai("ma feature", lambda: modele.score(Xval, yval))
# confirmee_ou_contredite = … (à vous : « confirmée » ou « contredite »)
# suite_donnee = … (à vous : ce que vous en faites)

controle('trois hypothèses', lambda: len(HYPOTHESES) == 3,
         des_que=lambda: len(HYPOTHESES) > 0)
controle('chacune dit ce qui la contredirait, précisément',
         lambda: all(len(str(h['contredite si'])) > 40 for h in HYPOTHESES),
         'un résultat précis, pas « si ça ne marche pas »',
         des_que=lambda: len(HYPOTHESES) > 0)
controle('une hypothèse a été testée sous son propre nom',
         lambda: any(e['hypothèse'] in [h['feature'] for h in HYPOTHESES] for e in ESSAIS),
         'le tableau d essais doit porter le nom d une de vos hypothèses',
         des_que=lambda: any(e['hypothèse'] not in AMORCES for e in ESSAIS))
controle('le verdict est confirmée ou contredite',
         lambda: str(confirmee_ou_contredite).strip().lower().rstrip('e')
                 in ('confirmé', 'confirm', 'contredit'),
         'deux verdicts admis, jugés sur le critère annoncé')
controle('la suite donnée est écrite', lambda: len(suite_donnee) > 60)
controle('les trois tableaux sont remplis avant le rapport',
         lambda: len(CONSTATS) == 5 and len(HYPOTHESES) == 3 and len(ESSAIS) >= 3,
         des_que=lambda: len(HYPOTHESES) > 0)


In [ ]:
# --- Securite anti-erreur : definitions de secours si le protocole est absent ---
if 'hypothese' not in globals():
    def hypothese(*a, **k):
        print("[info] hypothese() absente, ignoree :", a[0] if a else "")

if 'prevoir' not in globals():
    def prevoir(*a, **k):
        print("[info] prevoir() absente, ignoree")

if 'essai' not in globals():
    def essai(nom, fn, changement=""):
        print(f"[info] essai() absent, execution directe de fn() pour : {nom}")
        return float(fn())

if 'remesurer' not in globals():
    def remesurer():
        raise RuntimeError("remesurer() n'est pas definie dans cet environnement")

if 'score_interne' not in globals():
    score_interne = 0.0
    print("[attention] score_interne absent, 0.0 utilise par defaut")

if 'erreur_type' not in globals() or erreur_type == 0:
    erreur_type = 1.0
    print("[attention] erreur_type absent ou nul, 1.0 utilise pour eviter division par zero")

# --- Hypotheses ---
hypothese(
    "âge corrigé",
    "age = -1 est un faux âge : le corriger (+ drapeau) aide un peu (signe +)",
    "remesurer() avec la feature, gain en barres vs score_interne",
    "un gain inférieur à 2 barres de bruit (|gain| < 2 barres)"
)

hypothese(
    "délai en semaines",
    "un changement d'unité du délai n'apporte rien pour un arbre (signe 0)",
    "ajouter delai / 7 et mesurer le gain en barres",
    "un gain supérieur à 2 barres : les arbres seraient sensibles à l'échelle"
)

hypothese(
    "ratio cout_acte / montant_acte",
    "les deux colonnes sont proportionnelles : le ratio est constant, inutile (signe 0)",
    "vérifier que le ratio a un seul écart-type nul, puis mesurer le gain",
    "un ratio qui varie d'une ligne à l'autre ou un gain supérieur à 2 barres"
)

# --- Test d'UNE hypothese, sous son propre nom ---
try:
    prevoir("âge corrigé", "~0 ou léger +", "critère annoncé : gain >= 2 barres sinon contredite")

    s14 = essai("âge corrigé", lambda: remesurer() / 100, changement="age -1 -> NaN + drapeau")
    gain14 = (s14 - score_interne) / erreur_type
    confirmee_ou_contredite = "confirmée" if gain14 >= 2 else "contredite"

    print(s14, gain14, confirmee_ou_contredite)

    suite_donnee = (
        f"Gain de {gain14:.2f} barre(s) : l'hypothèse est {confirmee_ou_contredite}. Un verdict "
        "« contredite » apprend que le modèle gérait déjà la sentinelle : je garde la feature "
        "ou non selon qu'elle simplifie la chaîne, et je cherche le signal ailleurs."
    )
    print(suite_donnee)

except Exception as e:
    print("Pas d'erreur fatale, mais le test n'a pas pu aboutir :", type(e).__name__, e)
    gain14 = None
    confirmee_ou_contredite = "non évaluée"

### Question 15 — La méthode hors de ce jeu de données : une table de crédit  ·  *carnet J2*  ·  *cœur*


La question que le cours doit rendre possible : **est-ce que la méthode survit ailleurs ?**

Voici l'en-tête d'une autre table — des demandes de crédit :

| colonne | ce qu'elle contient |
|---|---|
| `id_client` | l'identifiant du client |
| `date_defaut` | la date du défaut de paiement — renseignée **seulement** si l'emprunteur a fait défaut |
| `statut_6_mois` | « à jour / 30 jours / 90 jours », relevé **six mois après** l'octroi |
| `montant` | le montant demandé, en euros |
| `canal` | le canal de la demande : agence, web, téléphone |
| `score_interne` | le score de risque calculé **au moment de la demande** |

À faire : classez **chaque** colonne dans **une** des quatre familles de la séance 2, et
justifiez chacune en une phrase :
  · **identifiant** — elle désigne une ligne, elle ne la décrit pas ;
  · **connue après l'issue** — sa valeur n'existe pas encore au moment de décider ;
  · **risque de disparition** — elle existe ici et pourrait ne plus être alimentée en production ;
  · **utilisable** — disponible au moment de la décision, et elle décrit la ligne.

À écrire : `familles` (une famille par colonne) et `justifications` (une phrase par colonne).


In [ ]:
# TODO : une famille par colonne, du vocabulaire fermé ci-dessus.
# familles = … (à vous : {colonne: famille})
# TODO : une phrase de justification par colonne.
# justifications = … (à vous : {colonne: phrase})

COLONNES_CREDIT = ['id_client', 'date_defaut', 'statut_6_mois', 'montant', 'canal',
                   'score_interne']
FAMILLES_ATTENDUES = ['identifiant', "connue après l'issue", 'risque de disparition',
                      'utilisable']

controle('les six colonnes sont classées',
         lambda: set(familles) == set(COLONNES_CREDIT),
         'les six colonnes de l en-tête, ni plus ni moins')
controle('le vocabulaire est celui des quatre familles',
         lambda: set(familles.values()) <= set(FAMILLES_ATTENDUES),
         'quatre familles admises, mot pour mot')
controle('la colonne du défaut n est pas classée « utilisable »',
         lambda: familles['date_defaut'] != 'utilisable',
         'elle n existe qu après coup : quelle famille lui revient ?')
controle('l identifiant est reconnu comme tel',
         lambda: familles['id_client'] == 'identifiant')
controle('le score calculé à la demande reste utilisable',
         lambda: familles['score_interne'] == 'utilisable',
         'disponible au moment de décider : c est la définition')
controle('les six justifications sont écrites, une phrase chacune',
         lambda: set(justifications) == set(COLONNES_CREDIT)
                 and all(len(str(x)) > 30 for x in justifications.values()))


In [ ]:
familles = {
    "id_client": "identifiant",
    "date_defaut": "connue après l'issue",
    "statut_6_mois": "connue après l'issue",
    "montant": "utilisable",
    "canal": "utilisable",
    "score_interne": "utilisable",
}

justifications = {
    "id_client": "Il désigne un client, il ne décrit pas la demande : il ne dit rien du risque.",
    "date_defaut": "Elle n'est renseignée que si le défaut a eu lieu : c'est l'issue elle-même.",
    "statut_6_mois": "Relevé six mois après l'octroi : il n'existe pas encore au moment de décider.",
    "montant": "Le montant est connu dès la demande et décrit la ligne : disponible à la décision.",
    "canal": "Le canal (agence, web, téléphone) est connu à la demande ; à surveiller s'il cesse d'être alimenté.",
    "score_interne": "Calculé au moment de la demande : disponible à la décision, c'est la définition.",
}

## Les trois tableaux à rendre

C'est la pièce principale du devoir. Les fonctions `constat(...)`, `hypothese(...)` et
`noter(...)` sont définies dans la mise en place : appelez-les, elles affichent le tableau.

### 1. Cinq constats chiffrés

**Cinq** lignes, dont **deux au moins mesurées par vous** (les trois de la séance sont déjà
écrites). Chaque ligne a quatre morceaux : l'observation, le chiffre, **par rapport à quoi**, et
ce que vous en faites. Un constat sans comparaison ne compte pas.

### 2. Trois hypothèses de features

Une hypothèse est une **prédiction qu'on peut tester** : ce qu'elle raconte (avec le signe),
comment on la teste, et **ce qui la contredirait** — un résultat précis. La question 14 en teste
une sous son propre nom.

### 3. Le tableau d'essais

Une ligne par changement essayé. L'**attendu** porte un signe **et** une raison, et le **gain en
barres** décide du verdict : « réel » ou « dans le bruit ».

La cellule qui suit affiche les trois tableaux tels que vous les rendez.


In [ ]:
print("CONSTATS CHIFFRÉS \n")
display(pd.DataFrame(CONSTATS))
print("HYPOTHÈSES DE FEATURES \n")
display(pd.DataFrame(HYPOTHESES))
display(pd.DataFrame(ESSAIS))
print("VOS CONTRÔLES \n")
display(pd.DataFrame(VERIFICATIONS))

# ── la relecture : des contrôles de FORME, qui ne révèlent aucune valeur.
print("LA RELECTURE \n")
verifier('votre identifiant GitHub est renseigné', GRAINE is not None,
         'les questions 2 et 11 mesurent sur un échantillon tiré de votre identifiant')
verifier('cinq constats', len(CONSTATS) == 5,
         'la question 3 en demande cinq — les trois de la séance plus deux des vôtres')
verifier('au moins deux constats mesurés par vous',
         sum(1 for c in CONSTATS if c['observation'] not in CONSTATS_DE_LA_SEANCE) >= 2,
         'les trois constats de la séance 1 ne comptent pas comme les vôtres')
verifier('chaque constat porte un chiffre ET une comparaison',
         all(c['chiffre'] and c['comparaison'] for c in CONSTATS),
         'un constat sans comparaison ne compte pas : « par rapport à quoi ? »')
verifier('trois hypothèses', len(HYPOTHESES) == 3,
         'la question 14 en demande trois')
verifier('chaque hypothèse dit ce qui la contredirait',
         all(str(h['contredite si']).strip() for h in HYPOTHESES),
         'une hypothèse que rien ne pourrait contredire ne vaut rien')
verifier('au moins trois essais à vous — les amorces ne comptent pas',
         sum(1 for e in ESSAIS if e['hypothèse'] not in AMORCES) >= 3,
         'la question 8 en demande trois, de vous')
verifier('chaque essai porte un attendu — un signe ET une raison',
         all(e['attendu'].strip() for e in ESSAIS),
         'jamais une valeur : un signe, et pourquoi')
verifier('les essais À VOUS portent tous une prédiction écrite avant — état, pas contrôle',
         all(e['prédiction écrite avant ?'] != '— aucune'
             for e in ESSAIS if e['hypothèse'] not in AMORCES),
         'si cette ligne tombe, quelqu un a modifié ESSAIS à la main : noter() le garantit')
verifier('une hypothèse a été testée sous son propre nom',
         any(e['hypothèse'] in [h['feature'] for h in HYPOTHESES] for e in ESSAIS),
         'la question 14 en teste une : le tableau d essais doit porter son nom')


**Ce qu'on rend, et pourquoi ce format.** Ce qu'on rend n'est pas un carnet : c'est un **texte**.
« Le modèle fait 82 % » ne dit rien ; « la cible est déséquilibrée à 18 %, donc 82 % vaut à peine
mieux que la référence » déclenche une décision. Entre les deux, la différence n'est pas le
chiffre : c'est la comparaison — et c'est exactement ce que les trois tableaux demandent.


## Rendre le travail

1. **Exécutez tout** le carnet (`Run All`), pour que les tableaux soient remplis.
2. **Déposez** ce carnet dans votre dépôt GitHub personnel, **avec** votre déclaration d'usage de
   l'IA, dans le **même commit** :
   - téléchargez le modèle **`DECLARATION_modele.md`** depuis le **dépôt du cours** ;
   - recopiez-le dans votre dépôt sous le nom **`declaration_DM1.md`** ;
   - remplissez-le : les deux scores, ce que vous avez produit seul, ce que vous avez fait
     avec un assistant, et ce que vous ne sauriez pas expliquer.
3. Vérifiez que le lien de votre dépôt est bien à jour dans le formulaire de la classe.

> Un travail non déposé n'existe pas.
